# STORY wearable analysis

Set paths and analysis options in the config cell, then run top to bottom. To rebuild the Ōura and ED-15 input files from raw data first, set `REBUILD_OURA_FROM_RAW` / `REBUILD_ARMT_FROM_RAW` to `True` (Appendices A and B).

# Configuration

In [ ]:
from pathlib import Path

# Run mode
REBUILD_OURA_FROM_RAW = False   # True = re-read Ōura server folders. False = load daily file built by a previous run.
REBUILD_ARMT_FROM_RAW = False   # True = rebuild armt_scores.csv from RADAR (Appendix B).

# Person-specific models
PERSON_FIXED_C  = 1.0
PERSON_FIXED_L1 = 0.5

# Paths
DATA_DIR = Path("path/to/data")      # cohort, ED-15 and Ōura daily files

OURA_RAW_DIR   = Path("path/to/oura_raw/")       # server, needs VPN
RADAR_RAW_DIR  = Path("path/to/radar_raw")   # server, needs VPN if wfh
COHORT_CSV     = DATA_DIR / "STORY_clean_20260813.csv"
ED15_CSV       = DATA_DIR / "armt_scores.csv"
OURA_DAILY_CSV = DATA_DIR / "oura_daily_clean.csv"
OUT_DIR        = Path("path/to/output")

# Cohort freeze
DATA_CUTOFF = "2026-07-31"

# Healthy controls
# WARNING! will inflate pooled discrimination because model can learn ED vs HC using BMI, sleep etc. Always read section 7 alongside.
INCLUDE_CONTROLS = False

# Eligibility
MIN_N_ED15 = 5                  # minimum ED-15 completions per participant

# ED-15 total as CHANGE, not level
ED15_CLINICAL_CUTOFF  = 2.1   # >90% of windows above clinical threshold. Modelling direction of change as alternative.
MODEL_ED15_CHANGE     = True
ED15_CHANGE_MIN_DELTA = 0.0   # |change| must exceed this to count; 0.0 = any change
ED15_CHANGE_DROP_TIES = True  # dropping exact no-change completions rather than call them "no"

# ED-15 build options (Appendix B)
EXCLUDE_PRACTICE_ROUNDS  = True   # exclude completions within PRACTICE_ROUND_DAYS of onboarding
PRACTICE_ROUND_DAYS      = 7
REQUIRE_COMPLETE_GAD_PHQ = True   # True = partly answered GAD-7/PHQ-8 becomes missing
ED15_MIN_ATTITUDINAL     = 8      # of 10 items needed for a total score
COMPANION_FILE_TOL_DAYS  = 3      # GAD-7/PHQ-8/weight files may be dated +/- 3 days of completing ED15

# Ōura wear validity
MAX_NON_WEAR_SECONDS = 28_800 # weartime >16 hrs/day

# Analysis windows
# (days_before_start, days_before_end, minimum valid days required). Note: ED-15 behavioural items are 1-week recall.
WINDOWS = {
    "full14":     (14, 1, 10),   # primary
    "antecedent": (14, 8,  5),   # sensitivity - entirely before the recall week
}
PRIMARY_WINDOW = "full14"

PREDICT_NEXT_ED15 = True        # predict ED-15 after sensor fortnight; windows are paired to next completion
NEXT_ED15_GAP     = (11, 17)    # days between consecutive completions

# Feature construction
ALLOWED_SUFFIXES = ("_mean", "_std", "_slope", "_range", "_max_change", "_pct_extreme")
MAX_MISSINGNESS  = 0.50            # drop features missing in >50% of windows
EXCLUDE_PREFIXES = ("vo2_max_",)   # VO2max only present for <10% of participants

# Modelling
SPLIT_FRACTIONS   = [0.40, 0.60, 0.80]

# Personal-baseline (within-person z-score) settings
Z_METHOD_PRIMARY = "none"           # pooled models: standardise across training set
Z_METHOD_PERSON  = "trainfold"      # person-specific models: within-participant
Z_METHODS        = ["none", "trainfold"]   # the two options; only the primary is run
N_BOOTSTRAP_Z  = 500
PARAM_GRID = {"clf__C": [0.01, 0.1, 1.0, 10.0], "clf__l1_ratio": [0.2, 0.5, 0.8]}
MIN_TRAIN_WINDOWS = 3          # per person, for person-specific model

# Rare categorical levels
MIN_CATEGORY_PARTICIPANTS = 5   # baseline categories with fewer participants collapsed into "Other"

# Baseline variables to keep in the descriptives, NOT in models. 
DROP_FROM_MODELS = ["gender", "ed_diagnosis_current_clean", "pt_final"]

# Warning for baseline variable missing for more than x-fraction of participants
WARN_BASELINE_MISSING = 0.50  # consider removing so doesn't enter model silently, median-imputed for most of the sample

# Three validation options (section 9)
THREE_WAY_FEATURE_SET = "all"   # A=cohort only (predict for new patient), B=person only, C=add personal to cohort as accumulates
THREE_WAY_N_COHORT_FOLDS = 5
MIN_TEST_WINDOWS  = 5           # per person, before summarising person-specific performance
N_BOOTSTRAP       = 2000        # participant-level cluster bootstrap for CIs
N_PERM_REPEATS    = 20
RANDOM_STATE      = 42

SECONDS_PER_HOUR = 3600.0
print("Configuration loaded.")
print(f"  Primary window = {PRIMARY_WINDOW}: days -{WINDOWS[PRIMARY_WINDOW][0]} to -{WINDOWS[PRIMARY_WINDOW][1]}")

## Imports and small helpers

`section()` and `note()` just make the output readable. `check()` prints a pass/fail line
for each assumption so you can see at a glance whether a decision held.

In [ ]:
import warnings, json, textwrap, os, re, gzip
import numpy as np
import pandas as pd
from scipy.stats import linregress
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score,
                             precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

from sklearn.exceptions import ConvergenceWarning, UndefinedMetricWarning
for _w in (FutureWarning, UserWarning, ConvergenceWarning, UndefinedMetricWarning):
    warnings.filterwarnings("ignore", category=_w)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

DATA_CUTOFF_TS = pd.Timestamp(DATA_CUTOFF)
RESULTS = {}          # everything worth reporting gets stashed here for Appendix C

def section(title):
    """Big visual header."""
    print("\n" + "=" * 78)
    print(title.upper())
    print("=" * 78)

def note(text):
    """Plain-English explanation printed above a block of output."""
    print(textwrap.fill(text, 76, initial_indent="  ", subsequent_indent="  "))
    print()

def check(label, passed, detail=""):
    """Print a pass/fail line for an assumption."""
    mark = "PASS" if passed else "**FAIL**"
    print(f"  [{mark}] {label}" + (f"  -  {detail}" if detail else ""))
    return passed

def hrs(seconds):
    return pd.to_numeric(seconds, errors="coerce") / SECONDS_PER_HOUR

print("Libraries loaded.")

# 1. Ōura data

Creates file with **one row per participant per day**, frozen at extraction data (`DATA_CUTOFF`). Every table is collapsed to one row per person-day before merging (deduplicating sleep rows), then `assert` after each merge that the row count did not change.

Prints overview of valid days based on `MAX_NON_WEAR_SECONDS` set in the config cell

Inspect printed summary table at the end for out of range values and consider implementing lower/upper bounds for implausible or impossible values.

In [ ]:
# 1.1 - Load
section("1. Ōura data")

oura_daily = pd.read_csv(OURA_DAILY_CSV, low_memory=False)
oura_daily["day"] = pd.to_datetime(oura_daily["day"], errors="coerce")

ed15 = pd.read_csv(ED15_CSV, low_memory=False)
ed15["date"] = pd.to_datetime(ed15["date"], errors="coerce")

baseline = pd.read_csv(COHORT_CSV, low_memory=False)

for _d in (oura_daily, ed15, baseline):
    _d["story_id"] = pd.to_numeric(_d["story_id"], errors="coerce").astype("Int64")

print(f"Ōura daily rows : {len(oura_daily):,}  ({oura_daily['story_id'].nunique()} participants)")
print(f"ED-15 rows      : {len(ed15):,}  ({ed15['story_id'].nunique()} participants)")
print(f"Baseline rows   : {len(baseline):,}")

# does the cached Ōura file actually reach the cut-off
_maxday = oura_daily["day"].max()
if pd.notna(_maxday) and _maxday < DATA_CUTOFF_TS:
    print(f"\n  NOTE: {OURA_DAILY_CSV.name} ends {_maxday.date()}, before the cut-off "
          f"({DATA_CUTOFF_TS.date()}).")
    print(f"        Re-run Appendix A to extend it, or move DATA_CUTOFF back.")

display(oura_daily.head(2))
display(ed15.head(2))

# 1.2 - Cohort freeze and duplicate check
_n0_oura, _n0_ed15 = len(oura_daily), len(ed15)
oura_daily = oura_daily[oura_daily["day"] <= DATA_CUTOFF_TS].copy()
ed15       = ed15[ed15["date"] <= DATA_CUTOFF_TS].copy()
print(f"\n  Ōura rows : {_n0_oura:,} -> {len(oura_daily):,}")
print(f"  ED-15 rows: {_n0_ed15:,} -> {len(ed15):,}\n")

_n1 = len(oura_daily)
oura_daily = oura_daily.drop_duplicates()
_exact = _n1 - len(oura_daily)

_dupe_days = oura_daily.duplicated(subset=["story_id", "day"]).sum()
if _dupe_days:
    # keep the row with the LOWEST non-wear time = the most complete sync for that day
    oura_daily = (oura_daily.sort_values("non_wear_time")
                            .drop_duplicates(subset=["story_id", "day"], keep="first")
                            .sort_values(["story_id", "day"]))

print("Duplicate handling")
print(f"  exact duplicate rows removed        : {_exact:,}")
print(f"  person-days with >1 row, resolved   : {_dupe_days:,}")
print(f"  (resolved by keeping the row with the lowest non-wear time)\n")

check("one row per participant-day",
      not oura_daily.duplicated(subset=["story_id", "day"]).any(),
      f"{len(oura_daily):,} rows, {oura_daily['story_id'].nunique()} participants")

RESULTS["n_oura_days"] = len(oura_daily)
RESULTS["n_oura_participants"] = int(oura_daily["story_id"].nunique())

# 1.3 - Wear validity
oura_daily["day_valid"] = (
    pd.to_numeric(oura_daily["non_wear_time"], errors="coerce") < MAX_NON_WEAR_SECONDS
).fillna(False)

print()
note(f"A day counts as VALID if non-wear time is under {MAX_NON_WEAR_SECONDS:,} s "
     f"({MAX_NON_WEAR_SECONDS/3600:.1f} h non-wear, i.e. at least "
     f"{24 - MAX_NON_WEAR_SECONDS/3600:.1f} h of wear).")
print(f"  Valid days: {int(oura_daily['day_valid'].sum()):,} of {len(oura_daily):,} "
      f"({100*oura_daily['day_valid'].mean():.1f}%)\n")

RESULTS["pct_valid_days"] = float(100 * oura_daily["day_valid"].mean())

# 1.4 - Summaries for inspection
_num = [c for c in oura_daily.columns
        if c not in ("story_id", "day", "day_valid")
        and pd.api.types.is_numeric_dtype(oura_daily[c])]

_rows = []
for c in _num:
    v  = pd.to_numeric(oura_daily[c], errors="coerce")
    ok = v.dropna()
    if ok.empty:
        continue
    q1, q3 = ok.quantile(.25), ok.quantile(.75)
    iqr    = q3 - q1
    lo, hi = q1 - 3 * iqr, q3 + 3 * iqr          # 3xIQR = far out, not merely unusual
    _rows.append({
        "variable": c,
        "n": len(ok),
        "%miss": 100 * v.isna().mean(),
        "min": ok.min(), "p1": ok.quantile(.01), "median": ok.median(),
        "p99": ok.quantile(.99), "max": ok.max(),
        "n_far_out": int(((ok < lo) | (ok > hi)).sum()),
        "%far_out": 100 * ((ok < lo) | (ok > hi)).mean()})

summary = pd.DataFrame(_rows).sort_values("%far_out", ascending=False)

note("One row per daily Ōura variable. Note: 'far out' = beyond 3x IQR. Consider removing.")
with pd.option_context("display.width", 220, "display.max_rows", 200):
    print(summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

print("\n\nSleep variables in hours (inspect for out-of-range)")
for c in ["total_sleep_duration", "main_sleep_duration", "time_in_bed",
          "light_sleep_duration", "deep_sleep_duration", "rem_sleep_duration",
          "awake_time", "nap_total_duration"]:
    if c not in oura_daily.columns:
        continue
    h = hrs(oura_daily[c]).dropna()
    if h.empty:
        continue
    print(f"  {c:<24} median {h.median():>6.2f}  p99 {h.quantile(.99):>6.2f}  "
          f"max {h.max():>7.2f}   >14h: {int((h > 14).sum()):>5}   >24h: {int((h > 24).sum()):>4}")

print("\nVariables with high missingness")
_m = summary.nlargest(8, "%miss")[["variable", "%miss", "n"]]
print(_m.to_string(index=False, float_format=lambda x: f"{x:,.1f}"))

RESULTS["oura_summary"] = summary

# 2. ED-15 outcomes

Three binary behavioural outcomes - behaviour present vs absent in the past 7 days:
- **Restrict** - `ed15_restrict > 0`
- **Binge** - `ed15_binge > 0`
- **Purge** - any of `ed15_exercise`, `ed15_vomit`, `ed15_laxatives` > 0

Plus one change outcome:
- **Deteriorate** - ED-15 total score went **up** vs **down** since the previous completion. Completions with no change are dropped (`ED15_CHANGE_DROP_TIES`), so this is an increase-vs-decrease contrast, not increase-vs-no-increase.

**The ED-15 total score is not modelled as a level**, only as direction of change: >90% of windows exceeded the clinical threshold, so the level barely varies.

**Note**: `n_varies` counts participants whose outcome is **not constant** across their assessments.
If this number is small, report as important constraint in analysis.

In [ ]:
section("2. ED-15 outcomes")

OUTCOMES = ["Restrict", "Binge", "Purge"]
CHANGE_COVARIATES = ["ed15_total_prev", "ed15_gap_days"]

ed15["Restrict"] = (pd.to_numeric(ed15["ed15_restrict"], errors="coerce") > 0).astype(float)
ed15["Binge"]    = (pd.to_numeric(ed15["ed15_binge"],    errors="coerce") > 0).astype(float)
ed15["Purge"]    = (ed15[["ed15_exercise", "ed15_vomit", "ed15_laxatives"]]
                    .apply(pd.to_numeric, errors="coerce").gt(0).max(axis=1).astype(float))

# ED-15 total change (direction), computed over ALL completions
# "Previous" means the previous completion, whether or not that window had
# usable Ōura data - so the change is a property of the questionnaire series.
if MODEL_ED15_CHANGE and "ed15_total_score" in ed15.columns:
    ed15 = ed15.sort_values(["story_id", "date"]).reset_index(drop=True)
    g = ed15.groupby("story_id")
    ed15["ed15_total_prev"] = g["ed15_total_score"].shift(1)
    ed15["ed15_gap_days"]   = g["date"].diff().dt.days
    _delta = ed15["ed15_total_score"] - ed15["ed15_total_prev"]
    ed15["ed15_delta"] = _delta
    _up   = _delta >  ED15_CHANGE_MIN_DELTA
    _down = _delta < -ED15_CHANGE_MIN_DELTA
    ed15["Deteriorate"] = np.where(_up, 1.0, np.where(_down, 0.0, np.nan if ED15_CHANGE_DROP_TIES else 0.0))
    ed15.loc[_delta.isna(), "Deteriorate"] = np.nan
    OUTCOMES = OUTCOMES + ["Deteriorate"]
    _n_first = int(_delta.isna().sum())
    _n_tie   = int(((~_up) & (~_down) & _delta.notna()).sum())
    print(f"ED-15 change outcome built ('Deteriorate' = total score went up)")
    print(f"   first completion per person (no previous score) : {_n_first:,} -> missing")
    print(f"   no change beyond +/-{ED15_CHANGE_MIN_DELTA}                          : {_n_tie:,}"
          f" -> {'dropped' if ED15_CHANGE_DROP_TIES else 'counted as no increase'}")
    print(f"   usable                                          : "
          f"{int(ed15['Deteriorate'].notna().sum()):,}\n")

# eligibility: enough completions to model
_counts  = ed15.groupby("story_id").size()
eligible = _counts[_counts >= MIN_N_ED15].index
ed15_el  = ed15[ed15["story_id"].isin(eligible)].copy()

note(f"Participants need at least {MIN_N_ED15} ED-15 completions to enter the analysis.")
print(f"  ED-15 completions total                    : {len(ed15):,}")
print(f"  Participants with >= {MIN_N_ED15} completions        : {ed15_el['story_id'].nunique()}")
print(f"  Completions retained                       : {len(ed15_el):,}\n")

print(f"{'outcome':<10} {'% positive':>11} {'n varies':>10} {'n always +':>11} {'n always -':>11}")
print("-" * 58)
_out_rows = []
for o in OUTCOMES:
    g = ed15_el.groupby("story_id")[o]
    n_var    = int((g.nunique() == 2).sum())
    n_always = int((g.mean() == 1).sum())
    n_never  = int((g.mean() == 0).sum())
    print(f"{o:<10} {100*ed15_el[o].mean():>10.1f}% {n_var:>10} {n_always:>11} {n_never:>11}")
    _out_rows.append({"outcome": o, "pct_positive": 100*ed15_el[o].mean(),
                      "n_varies": n_var, "n_always": n_always, "n_never": n_never})

print()

RESULTS["outcome_variation"] = pd.DataFrame(_out_rows)

# 3. Build the analysis windows

For each ED-15 completion, wearable data is summarised over a window **before**. There are three possible windows:

```
                                 ED-15 (A)             ED-15 (B)
day:  -14 -13 ... -8 | -7 -6 ... -1 |                   |
                      \_____________/
                       week ED-15 asks about
      |<------ full14 (primary) --->|X                  |
      |<-antecedent->|               X                  |
      |<-------- full14next ------->|                   |X
```

- **`full14` (−14 to −1)** Half sits inside recall period -> Concurrent. 
- **`antecedent` (−14 to −8)** *before* recalled week -> Prospective (sensitivity analysis).
- **`full14_next` (-14..-1)** predicts *next* questionnaire (B) -> Prospective (sensitivity analysis).

Summary statistics per feature (set by `ALLOWED_SUFFIXES` in the config cell):
- `mean`, `std`, `range`, `max_change`.
- `slope` is fitted against the *actual day offset*, so missing days don't distort the trend.
- `pct_extreme` applies a 1.5×IQR rule to 6-14 points; close to noise at that window length.

**NOTE for `full14_next`:** Window is kept only where two consecutive completions are `NEXT_ED15_GAP` days apart and each participant's final completion has no successor and is dropped. Set `PREDICT_NEXT_ED15 = False` in the config cell to skip it.

In [ ]:
section("3. building analysis windows")

CARRY_COLS = [c for c in ["ed15_total_prev", "ed15_gap_days", "ed15_delta"]
              if c in ed15_el.columns]

FEATURES = [f for f in [
    "activity_score","steps","equivalent_walking_distance","total_calories","active_calories",
    "average_met_minutes","high_activity_met_minutes","high_activity_time",
    "low_activity_met_minutes","low_activity_time","medium_activity_met_minutes",
    "medium_activity_time","resting_time","sedentary_met_minutes","sedentary_time",
    "time_in_bed","sleep_score","total_sleep_duration","light_sleep_duration",
    "deep_sleep_duration","rem_sleep_duration","main_sleep_duration",
    "main_sleep_midpoint_mins","main_bedtime_start_mins","nap_count","nap_total_duration",
    "social_jetlag_weekly","average_heart_rate","average_hrv","average_breath","efficiency",
    "lowest_heart_rate","restless_periods","latency","awake_time","temperature_deviation",
    "temperature_trend_deviation","oura_readiness_score","recovery_high","stress_high",
    "breathing_disturbance_index","spo2_percentage",
] if f in oura_daily.columns]

print(f"Daily features to summarise: {len(FEATURES)}\n")

STAT_NAMES = ["mean","std","min","max","last","range","max_change",
              "slope","median","iqr","pct_extreme"]

def summarise_window(values, day_offsets):
    """11 summary statistics."""
    out = {k: np.nan for k in STAT_NAMES}
    v = np.asarray(values, dtype=float)
    d = np.asarray(day_offsets, dtype=float)
    ok = ~np.isnan(v)
    v, d = v[ok], d[ok]
    if v.size == 0:
        return out
    out["mean"], out["std"] = v.mean(), (v.std(ddof=1) if v.size > 1 else np.nan)
    out["min"], out["max"]  = v.min(), v.max()
    out["last"]   = v[-1]
    out["range"]  = v.max() - v.min()
    out["median"] = float(np.median(v))
    out["max_change"] = float(np.abs(np.diff(v)).max()) if v.size > 1 else 0.0
    if v.size > 1 and np.ptp(d) > 0:
        out["slope"] = float(linregress(d, v).slope)      # per DAY
    else:
        out["slope"] = 0.0
    q1, q3 = np.percentile(v, 25), np.percentile(v, 75)
    out["iqr"] = q3 - q1
    if v.size >= 5:
        out["pct_extreme"] = float(((v < q1 - 1.5*out["iqr"]) |
                                    (v > q3 + 1.5*out["iqr"])).mean() * 100.0)
    return out

def longest_missing_run(valid_flags):
    run = best = 0
    for ok in valid_flags:
        run = 0 if ok else run + 1
        best = max(best, run)
    return best

def build_windows(oura_df, ed15_df, wname, verbose=True):
    """One row per ED-15 completion, with wearable summaries over the window."""
    d_start, d_end, min_valid = WINDOWS[wname]
    span = d_start - d_end + 1
    by_pid = {pid: g.sort_values("day").set_index("day")
              for pid, g in oura_df.groupby("story_id", sort=False)}

    rows, keep = [], []
    for _, r in ed15_df.iterrows():
        pid, eddate = r["story_id"], r["date"]
        days = pd.date_range(eddate - pd.Timedelta(days=d_start),
                             eddate - pd.Timedelta(days=d_end), freq="D")
        g = by_pid.get(pid)
        if g is None:
            rows.append({}); keep.append(False); continue

        win     = g.reindex(days)
        valid   = win["day_valid"].to_numpy(dtype=bool, na_value=False)
        n_valid = int(valid.sum())
        gap     = longest_missing_run(valid)
        keep.append(n_valid >= min_valid)

        s = {"n_valid_days": n_valid, "frac_valid_days": n_valid / span,
             "max_gap_days": gap,
             "mean_non_wear_time": win.loc[valid, "non_wear_time"].mean() if n_valid else np.nan}
        vw       = win[valid]
        offsets  = np.arange(span)[valid]
        for f in FEATURES:
            stats = summarise_window(vw[f].to_numpy() if f in vw.columns else [], offsets)
            for k, v in stats.items():
                s[f"{f}_{k}"] = v
        rows.append(s)

    summ = pd.DataFrame(rows).reset_index(drop=True)
    out  = pd.concat([ed15_df.reset_index(drop=True), summ], axis=1)
    out  = out[pd.Series(keep)].reset_index(drop=True)

    n_after = out.groupby("story_id").size()
    out = out[out["story_id"].isin(n_after[n_after >= MIN_N_ED15].index)].reset_index(drop=True)

    if verbose:
        print(f"  [{wname:<11}] days -{d_start} to -{d_end} | "
              f"{out['story_id'].nunique():>3} participants | {len(out):>4} windows "
              f"| median valid days {out['n_valid_days'].median():.0f}")
    return out.sort_values(["story_id", "date"]).reset_index(drop=True)

datasets = {w: build_windows(oura_daily, ed15_el, w) for w in WINDOWS}

# full14|next - predict the ED-15 that follows the sensor fortnight
def shift_to_next_ed15(df, gap=None):
    lo, hi = NEXT_ED15_GAP if gap is None else gap
    df = df.sort_values(["story_id", "date"]).copy()
    g = df.groupby("story_id")
    df["next_date"]    = g["date"].shift(-1)
    df["ed15_gap_days"] = (df["next_date"] - df["date"]).dt.days
    moved = []
    for col in OUTCOMES + [c for c in ["ed15_total_score"] if c in df.columns]:
        if col in df.columns:
            df[col] = g[col].shift(-1)
            moved.append(col)
    keep = df["ed15_gap_days"].between(lo, hi)
    out = df[keep].reset_index(drop=True)
    n_after = out.groupby("story_id").size()
    out = out[out["story_id"].isin(n_after[n_after >= MIN_N_ED15].index)].reset_index(drop=True)
    print(f"  [full14|next  ] outcome taken from the FOLLOWING completion "
          f"({lo}-{hi} days later)")
    print(f"                  {out['story_id'].nunique():>3} participants | "
          f"{len(out):>4} windows | dropped {len(df)-int(keep.sum())} unpaired "
          f"or out-of-range")
    return out

if PREDICT_NEXT_ED15:
    datasets["full14_next"] = shift_to_next_ed15(datasets["full14"])
print()
check(f"primary window '{PRIMARY_WINDOW}' produced data", len(datasets[PRIMARY_WINDOW]) > 0)
RESULTS["window_counts"] = pd.DataFrame([
    {"window": w, "participants": d["story_id"].nunique(), "windows": len(d),
     "median_valid_days": d["n_valid_days"].median()} for w, d in datasets.items()])

# 4. Baseline characteristics

Analysis is restricted to participants in the eating disorder group using `group`. `pt_final` is the phenotype variable and defines `group`, so will be collinear with 
`ed_diagnosis_current_clean` and `ed_type`.

Continuous baseline variables are reported as **mean (SD)** with range and n.

In [ ]:
section("4. baseline characteristics and sample")

BASE_NUM = ["age_numeric", "bmi_baseline", "illness_duration_years"]
BASE_CAT = ["pt_final", "gender", "ethnicity", "sex", 
            "ed_diagnosis_current_clean", "ed_type", "stage_label",
            "treatment_status_screening", "med_status"]
COVARIATES = [] 

_wanted = ["story_id", "group"] + BASE_NUM + BASE_CAT
_have   = [c for c in _wanted if c in baseline.columns]
_absent = [c for c in _wanted if c not in baseline.columns]

print("Cohort file check")
print(f"  columns found   : {[c for c in _have if c != 'story_id']}")
if _absent:
    print(f"  columns MISSING : {_absent}")

if "group" not in _have:
    raise KeyError(f"'group' is not in {COHORT_CSV.name}")
print()

def attach_baseline(df):
    """Merge baseline characteristics and keep only the eating disorder group."""
    n0, p0 = len(df), df["story_id"].nunique()

    already = [c for c in _have if c != "story_id" and c in df.columns]
    if already:
        df = df.drop(columns=already)

    m = df.merge(baseline[_have].drop_duplicates("story_id"),
                 on="story_id", how="left", validate="many_to_one")
    assert len(m) == n0, "baseline merge changed the row count"
    assert "group" in m.columns, "'group' vanished in the merge — check for a name collision"

    n_nogroup = int(m["group"].isna().sum())
    if INCLUDE_CONTROLS:
        m = m[m["group"].notna()].copy()
    else:
        m = m[m["group"] == "Eating disorder"].copy()
    return m, {"windows_before": n0, "participants_before": p0,
               "windows_after": len(m), "participants_after": m["story_id"].nunique(),
               "windows_no_group": n_nogroup}

_merge_log = {}
for w in list(datasets):
    datasets[w], _merge_log[w] = attach_baseline(datasets[w])

if INCLUDE_CONTROLS:
    print("!! If INCLUDE_CONTROLS = True, pooled AUC will rise because model learns ED vs HC from baseline.")
    print("   Behavioural outcomes near-constant; only 'Deteriorate. See within-person AUC in section 7.1.")

df_primary = datasets[PRIMARY_WINDOW]
_per = df_primary.groupby("story_id").size()
_fu  = df_primary.groupby("story_id")["date"].agg(lambda x: (x.max() - x.min()).days / 7)

print(f"  Participants                 : {df_primary['story_id'].nunique()}")
print(f"  Observation windows          : {len(df_primary):,}")
print(f"  Windows per person           : median {_per.median():.0f} "
      f"(IQR {_per.quantile(.25):.0f}-{_per.quantile(.75):.0f}, range {_per.min()}-{_per.max()})")
print(f"  Follow-up, weeks             : mean {_fu.mean():.1f} (SD {_fu.std():.1f}), "
      f"range {_fu.min():.0f}-{_fu.max():.0f}")
print(f"  Valid days per 14-day window : median {df_primary['n_valid_days'].median():.0f} "
      f"(IQR {df_primary['n_valid_days'].quantile(.25):.0f}-"
      f"{df_primary['n_valid_days'].quantile(.75):.0f})")

# guard: heavily-missing baseline variables
_bl1 = df_primary.drop_duplicates("story_id")
_hi_miss = []
for _c in BASE_NUM + BASE_CAT:
    if _c not in _bl1.columns: continue
    _f = float(_bl1[_c].isna().mean())
    if _f > WARN_BASELINE_MISSING:
        _hi_miss.append((_c, _f, int(_bl1[_c].isna().sum())))
if _hi_miss:
    print("\n  !! BASELINE VARIABLES ABOVE THE MISSINGNESS THRESHOLD")
    for _c, _f, _n in _hi_miss:
        _in = "EXCLUDED from models" if _c in DROP_FROM_MODELS else "still IN the models"
        print(f"     {_c:<28} {100*_f:>5.1f}% missing ({_n} participants)  -> {_in}")
    print("     Median imputation for most of the sample makes such a variable close to a")
    print("     constant, and it can act as a marker for who has incomplete records.")
    print("     Add it to DROP_FROM_MODELS in the config cell if that is not what you want.")

if DROP_FROM_MODELS:
    print(f"\n  Excluded from models (kept in descriptives): {DROP_FROM_MODELS}")

print("\n  Baseline characteristics:")
_bl = df_primary.drop_duplicates("story_id")
for c in BASE_NUM:
    if c in _bl.columns:
        v = pd.to_numeric(_bl[c], errors="coerce").dropna()
        if len(v):
            print(f"    {c:<26} mean {v.mean():.1f} (SD {v.std():.1f}), "
                  f"range {v.min():.1f}-{v.max():.1f}, n={len(v)}"
                  + (f", {len(_bl)-len(v)} missing" if len(v) < len(_bl) else ""))

# BMI by ED group
_grp = "ed_type"                    # or "ed_diagnosis_current_clean" / "pt_final"
if {"bmi_baseline", _grp} <= set(_bl.columns):
    print(f"    bmi_baseline by {_grp}:")
    for k, g in _bl.groupby(_grp, dropna=False)["bmi_baseline"]:
        v = pd.to_numeric(g, errors="coerce").dropna()
        if len(v):
            print(f"        {str(k):<24} mean {v.mean():.1f} (SD {v.std():.1f}), "
                  f"range {v.min():.1f}-{v.max():.1f}, n={len(v)}")
            
for c in ["gender", "sex", "ethnicity", "ed_type", "stage_label", "pt_final", 
          "ed_diagnosis_current_clean", "treatment_status_screening"]:
    if c in df_primary.columns:
        vc = _bl[c].value_counts(dropna=False)
        print(f"    {c}:")
        for k, v in vc.items():
            print(f"        {str(k):<24} {v:>3} ({100*v/vc.sum():.1f}%)")

RESULTS["n_participants"] = int(df_primary["story_id"].nunique())
RESULTS["n_windows"] = len(df_primary)

# Summary of ED-15 completions
_src = df_primary

def _row(col=None, thresh=0, purge=False):
    if purge:
        b = _src["Purge"].astype(bool)
        return {"Mean": "–", "SD": "–", "Median": "–", "Range": "–",
                "Frequency": "–", "% meeting threshold": f"{100*b.mean():.1f}"}

    if col == "Deteriorate":
        v = pd.to_numeric(_src["ed15_total_score"], errors="coerce").dropna()
        b = pd.to_numeric(_src["Deteriorate"], errors="coerce")
        return {"Mean": f"{v.mean():.2f}", "SD": f"{v.std():.2f}",
                "Median": f"{v.median():.1f}", "Range": f"{v.min():.0f}-{v.max():.0f}",
                "Frequency": f"{int(b.sum()):,}",
                "% meeting threshold": f"{100*b.mean():.1f}"}

    v = pd.to_numeric(_src[col], errors="coerce").dropna()
    b = pd.to_numeric(_src[col], errors="coerce") > thresh
    return {"Mean": f"{v.mean():.2f}", "SD": f"{v.std():.2f}",
            "Median": f"{v.median():.1f}", "Range": f"{v.min():.0f}-{v.max():.0f}",
            "Frequency": f"{int(b.sum()):,}", "% meeting threshold": f"{100*b.mean():.1f}"}

tbl2 = pd.DataFrame({
    "Total score": _row("Deteriorate"),
    "Restricting":     _row("ed15_restrict"),
    "Binging":         _row("ed15_binge"),
    "Purging":         _row(purge=True),
    "\u2003Exercising":   _row("ed15_exercise"),
    "\u2003Laxative use": _row("ed15_laxatives"),
    "\u2003Vomiting":     _row("ed15_vomit"),
}).T
tbl2.columns = pd.MultiIndex.from_tuples(
    [("Raw scores", c) for c in ["Mean", "SD", "Median", "Range"]] +
    [("Binarised metrics", c) for c in ["Frequency", "% positive"]])

print(f"\nTable 2. Summary of ED-15 completions (n = {len(_src):,}). "
      f"Total score: deterioration = increase from previous ED-15; "
      f"behaviours: positive = ≥1 occurrence.")
_d = pd.to_numeric(_src["Deteriorate"], errors="coerce")
print(f"  Total score: direction available for {int(_d.notna().sum()):,} "
      f"of {len(_src):,} windows.")
display(tbl2)
RESULTS["table2_ed15"] = tbl2

# 5. Feature sets

Five pre-specified sets (baseline-only, bl+activity, bl+physiology, bl+sleep, bl+all) to explore the incremental value of wearable data over baseline characteristics. Additional feature set wearable-only without baseline data to explore
if containing any signal.

**Rare categorical levels**: Levels held by fewer than `MIN_CATEGORY_PARTICIPANTS` participants are collapsed into "Other" before the feature sets are built. Collapsing is printed.

In [ ]:
section("5. feature sets")

ACTIVITY_BASES = ["activity_score","steps","equivalent_walking_distance","total_calories",
    "active_calories","average_met_minutes","high_activity_met_minutes","high_activity_time",
    "low_activity_met_minutes","low_activity_time","medium_activity_met_minutes",
    "medium_activity_time","resting_time","sedentary_met_minutes","sedentary_time"]
PHYSIO_BASES = ["average_heart_rate","average_hrv","average_breath","lowest_heart_rate",
    "temperature_deviation","temperature_trend_deviation","oura_readiness_score",
    "recovery_high","stress_high","breathing_disturbance_index","spo2_percentage"]
SLEEP_BASES = ["total_sleep_duration","light_sleep_duration","deep_sleep_duration",
    "rem_sleep_duration","efficiency","restless_periods","latency","awake_time","time_in_bed",
    "sleep_score","main_sleep_duration","main_sleep_midpoint_mins","main_bedtime_start_mins",
    "nap_count","nap_total_duration","social_jetlag_weekly"]

def pick_columns(df, bases, allowed=None):
    allowed = ALLOWED_SUFFIXES if allowed is None else allowed
    miss, out = df.isna().mean(), []
    for b in bases:
        for c in df.columns:
            if not c.startswith(b + "_"):        continue
            if c.endswith("_z"):                 continue   # never, under any setting
            if c.startswith(EXCLUDE_PREFIXES):   continue
            if not c.endswith(allowed):          continue
            if miss.get(c, 0.0) > MAX_MISSINGNESS: continue
            out.append(c)
    return sorted(set(out))

def build_feature_sets(df, allowed=None):
    A = pick_columns(df, ACTIVITY_BASES, allowed)
    P = pick_columns(df, PHYSIO_BASES,   allowed)
    S = pick_columns(df, SLEEP_BASES,    allowed)
    num_bl = [c for c in BASE_NUM if c in df.columns and c not in DROP_FROM_MODELS]
    cat_bl = [c for c in BASE_CAT if c in df.columns and c not in DROP_FROM_MODELS]
    cov    = [c for c in COVARIATES if c in df.columns]
    base   = num_bl + cat_bl + cov
    sets = {
        "baseline":      base,
        "bl_activity":   base + A,
        "bl_physiol":    base + P,
        "bl_sleep":      base + S,
        "all":           base + sorted(set(A + P + S)),
        "wearable_only": sorted(set(A + P + S)),
    }
    return sets, cat_bl

# collapse rare categorical levels before building feature sets
def collapse_rare_categories(df, cols, min_participants=None, verbose=True):
    min_participants = MIN_CATEGORY_PARTICIPANTS if min_participants is None else min_participants
    df = df.copy()
    if verbose:
        print(f"\nCollapsing categorical levels held by < {min_participants} participants:")
    for c in cols:
        if c not in df.columns: continue
        n_by_level = df.groupby(c)["story_id"].nunique()
        rare = n_by_level[n_by_level < min_participants].index.tolist()
        if rare:
            df[c] = df[c].where(~df[c].isin(rare), "Other")
            if verbose:
                detail = ", ".join(f"{lv} (n={n_by_level[lv]})" for lv in rare[:6])
                print(f"   {c:<28} {len(rare)} level(s) -> Other:  {detail}")
        elif verbose:
            print(f"   {c:<28} no change")
    return df

df_primary = collapse_rare_categories(df_primary, BASE_CAT)
for _w in datasets:
    datasets[_w] = collapse_rare_categories(datasets[_w], BASE_CAT, verbose=False)

FEATURE_SETS, CAT_COLS = build_feature_sets(df_primary)

# ED-15 total score change outcome gets two extra covariates: previous score and gap since it, 
# so the model doesn't simply predict on regression to mean. 
_extra = [c for c in CHANGE_COVARIATES if c in df_primary.columns]
FEATURE_SETS_CHANGE = {k: (v if k == "wearable_only" else v + _extra)
                       for k, v in FEATURE_SETS.items()}

def sets_for(outcome):
    """Feature sets appropriate to this outcome."""
    return FEATURE_SETS_CHANGE if outcome == "Deteriorate" else FEATURE_SETS

def feats_for(outcome, fname):
    return sets_for(outcome)[fname]

for k, v in FEATURE_SETS.items():
    print(f"  {k:<15} {len(v):>4} features"
          + (f"  (+{len(_extra)} for Deteriorate)" if _extra else ""))
print()
note("Feature counts are large relative to the number of participants. "
     "Repeated observations from one person carry less information than independent people"
     "so the effective sample size is closer to the participant count. This is why "
     "elastic net shrinks most coefficients to zero: see section 11 for how many.")
print(f"  effective n (participants) = {df_primary['story_id'].nunique()}   "
      f"features in 'all' = {len(FEATURE_SETS['all'])}")

# PERSONAL-BASELINE (WITHIN-PERSON z) HELPERS
Z_EXCLUDE = set(BASE_NUM + BASE_CAT + COVARIATES + CHANGE_COVARIATES)   # not z-scoring baseline variables

def zscoreable(feats):
    return [c for c in feats if c not in Z_EXCLUDE]

def apply_trainfold_z(tr, te, cols):
    tr, te = tr.copy(), te.copy()
    mu = tr.groupby("story_id")[cols].mean()
    sd = tr.groupby("story_id")[cols].std()
    sd = sd.where(sd > 1e-12)                    # constant in training -> undefined

    def _z(frame):
        m = frame[["story_id"]].join(mu, on="story_id")[cols]
        s = frame[["story_id"]].join(sd, on="story_id")[cols]
        out = (frame[cols].to_numpy(dtype=float) - m.to_numpy(dtype=float)) \
              / s.to_numpy(dtype=float)
        return out

    tr[cols] = _z(tr)
    te[cols] = _z(te)     # people absent from training -> all NaN, imputed downstream
    return tr, te

print(f"\nPersonal-baseline settings: methods = {Z_METHODS}, "
      f"primary = {Z_METHOD_PRIMARY}")

# 6. Splits and the model pipeline

* **The split**: Cuts are made *inside each participant's own timeline*: 40% / 60% / 80% of assessments train the model, the rest test it. The three cuts are nested (ie. the 80% test set sits inside the 60% one, which sits inside the 40% one). They are therefore reported separately, not averaged to not count later windows repeatedly. Read as dose-response in how much personal history is available (see section 9).
* **The pipeline**: Imputation and scaling sit **inside** the pipeline, so they are fitted on training data only.
* **Tuning**: `StratifiedGroupKFold` keeps each participant entirely within one inner fold, and the same person cannot appear in both inner-train and inner-validation.

In [ ]:
section("6. splits and model pipeline")

def person_cuts(df, frac):
    """Boolean mask: each participant's earliest `frac` of assessments -> training."""
    tr = pd.Series(False, index=df.index)
    for _pid, g in df.groupby("story_id"):
        g = g.sort_values("date")
        k = int(np.floor(len(g) * frac))
        if k >= 1:
            tr.loc[g.index[:k]] = True
    return tr

INNER_FRACTIONS = (0.6, 0.8)      # inner cuts, as a fraction of each person's TRAINING windows

def person_time_splits(story_ids, dates, fracs=INNER_FRACTIONS):
    """Inner splits drawn inside the TRAINING data only, per participant, forward in time.

    Each participant's earliest `frac` of their training windows fits the model; their
    remaining training windows check which settings work best. Nothing can cross the
    outer cut, because this function only ever sees training rows."""
    s = pd.DataFrame({"pid": np.asarray(story_ids),
                      "date": pd.to_datetime(np.asarray(dates))})
    s["pos"] = np.arange(len(s))
    out = []
    for f in fracs:
        tr, va = [], []
        for _pid, g in s.groupby("pid", sort=False):
            g = g.sort_values("date")
            k = int(np.floor(len(g) * f))
            tr.extend(g["pos"].to_numpy()[:k])
            va.extend(g["pos"].to_numpy()[k:])
        if len(tr) >= 5 and len(va) >= 5:
            out.append((np.asarray(tr), np.asarray(va)))
    return out

def make_pipeline(cat_cols, num_cols, scale_cols=None):
    """scale_cols: which numeric columns to standardise across participants.
    None means all of them. When wearable features have already been centred
    within each participant they are left alone, and only the baseline
    characteristics are scaled."""
    scale_cols = list(num_cols) if scale_cols is None else [c for c in scale_cols
                                                            if c in num_cols]
    plain_cols = [c for c in num_cols if c not in scale_cols]
    num_scaled = Pipeline([("impute", SimpleImputer(strategy="median")),
                           ("scale",  StandardScaler())])
    num_plain  = Pipeline([("impute", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="constant",
                                                  fill_value="Missing")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=2))])
    blocks = []
    if scale_cols: blocks.append(("num_scaled", num_scaled, scale_cols))
    if plain_cols: blocks.append(("num_plain",  num_plain,  plain_cols))
    if cat_cols:   blocks.append(("cat",        cat_pipe,   cat_cols))
    pre = ColumnTransformer(blocks, remainder="drop")
    clf = LogisticRegression(penalty="elasticnet", solver="saga", class_weight="balanced",
                             max_iter=5000, random_state=RANDOM_STATE)
    return Pipeline([("pre", pre), ("clf", clf)])

def fit_tuned(X, y, groups, dates, cat_cols, num_cols, scale_cols=None):
    """Elastic net, settings chosen by a time-ordered inner split within each participant.

    Nested: the outer cut (person_cuts) decides train vs test; this inner cut is drawn
    inside the training data only, so no test window influences the chosen settings."""
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) < 2:
        return None
    pipe = make_pipeline(cat_cols, num_cols, scale_cols)
    cv = person_time_splits(groups, dates)
    if not cv:                       # too little training data to split again
        pipe.set_params(clf__C=PERSON_FIXED_C, clf__l1_ratio=PERSON_FIXED_L1)
        pipe.fit(X, y)
        return pipe
    gs = GridSearchCV(pipe, PARAM_GRID, scoring="roc_auc", cv=cv,
                      n_jobs=-1, error_score=np.nan)
    gs.fit(X, y)
    return gs.best_estimator_

def fit_simple(X, y, cat_cols, num_cols, C=None, l1=None, scale_cols=None):
    """Fixed hyperparameters, used where there are too few windows to tune.

    NOTE: In scikit-learn, smaller C means stronger penalty.
    eg, C=0.1 harshest value and on standardised features shrinks every coefficient to zero
    when a person has only 3-8 training windows, leaving an intercept and constant prediction."""
    C  = PERSON_FIXED_C if C is None else C
    l1 = PERSON_FIXED_L1 if l1 is None else l1
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) < 2:
        return None
    pipe = make_pipeline(cat_cols, num_cols, scale_cols)
    pipe.set_params(clf__C=C, clf__l1_ratio=l1)
    pipe.fit(X, y)
    return pipe

def metrics(y, proba, threshold=0.5):
    y, proba = np.asarray(y, dtype=int), np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    prev = float(y.mean())
    out = {"n": len(y), "prevalence": prev,
           "precision": precision_score(y, pred, zero_division=0),
           "recall":    recall_score(y, pred, zero_division=0),
           "f1":        f1_score(y, pred, zero_division=0),
           "f1_all_positive": (2*prev/(1+prev)) if prev > 0 else np.nan}
    if len(np.unique(y)) == 2:
        out["auc"] = roc_auc_score(y, proba)
        out["ap"]  = average_precision_score(y, proba)
        out["ap_above_chance"] = out["ap"] - prev
    else:
        out.update({"auc": np.nan, "ap": np.nan, "ap_above_chance": np.nan})
    return out

_test_mask = ~person_cuts(df_primary, 0.80)
print(f"  At the 80% cut: {int((~_test_mask).sum()):,} training windows, "
      f"{int(_test_mask.sum()):,} test windows, "
      f"{df_primary.loc[_test_mask,'story_id'].nunique()} participants in test")
check("every test participant also appears in training",
      set(df_primary.loc[_test_mask,'story_id']) <= set(df_primary.loc[~_test_mask,'story_id']))

# 7. Pooled models

For each outcome and feature set, models are fitted at each of the three cuts and averaged across cuts, with confidence intervals from a participant-level cluster bootstrap. F1 is compared with `f1_trivial`, the F1 of a classifier that always predicts "yes".

Two benchmarks separate between-person from within-person signal: a base-rate benchmark (each participant's own historical rate, no wearable data) and within-person AUC (computed inside each participant, where the benchmark scores 0.5 by construction).

In [ ]:
section("7. pooled models")

def run_pooled(df, outcome, feats, cat_cols, n_boot=N_BOOTSTRAP, collect_preds=None,
               z_method="none", feature_label=None):
    """Fit at each cut, score each cut, average. Cluster-bootstrap CIs.

    z_method:
      "none"       use the raw feature values
      "trainfold"  standardise features against each person's TRAINING windows
    """
    feats = [c for c in feats if c in df.columns]
    cats  = [c for c in feats if c in cat_cols]
    nums  = [c for c in feats if c not in cats]
    zcols = zscoreable(nums) if z_method == "trainfold" else []
    d = df[df[outcome].notna()].copy()

    per_cut = []
    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        tr, te = d[trm], d[~trm]
        if tr.empty or te.empty:                 continue
        if tr[outcome].nunique() < 2:            continue
        if te[outcome].nunique() < 2:            continue
        if zcols:
            tr, te = apply_trainfold_z(tr, te, zcols)
        # wearable features are already on a within-person scale; scale only the rest
        _scale = [c for c in nums if c not in zcols] if zcols else None
        model = fit_tuned(tr[feats], tr[outcome], tr["story_id"], tr["date"], cats, nums,
                          scale_cols=_scale)
        if model is None:                        continue
        proba = model.predict_proba(te[feats])[:, 1]
        per_cut.append(pd.DataFrame({"frac": frac, "story_id": te["story_id"].to_numpy(),
                                     "y": te[outcome].astype(int).to_numpy(), "proba": proba}))
    if not per_cut:
        return None, None
    preds = pd.concat(per_cut, ignore_index=True)
    if collect_preds is not None:
        collect_preds.append(preds.assign(outcome=outcome, features=feature_label))

    def _avg(p):
        aucs, aps = [], []
        for _f, g in p.groupby("frac"):
            if g["y"].nunique() < 2: continue
            aucs.append(roc_auc_score(g["y"], g["proba"]))
            aps.append(average_precision_score(g["y"], g["proba"]))
        return (np.mean(aucs), np.mean(aps)) if aucs else (np.nan, np.nan)

    auc_hat, ap_hat = _avg(preds)
    base = metrics(preds["y"], preds["proba"])

    rng, bs = np.random.default_rng(RANDOM_STATE), []
    ids = preds["story_id"].unique()
    groups = {i: preds[preds["story_id"] == i] for i in ids}
    for _ in range(n_boot):
        draw = rng.choice(ids, size=len(ids), replace=True)
        bp = pd.concat([groups[i] for i in draw], ignore_index=True)
        a, _p = _avg(bp)
        if not np.isnan(a): bs.append(a)
    lo, hi = (np.percentile(bs, [2.5, 97.5]) if len(bs) >= 50 else (np.nan, np.nan))

    return {"outcome": outcome, "n_features": len(feats), "n": base["n"],
            "prevalence": base["prevalence"], "auc": auc_hat, "auc_lo": lo, "auc_hi": hi,
            "ap": ap_hat, "ap_above_chance": ap_hat - base["prevalence"],
            "precision": base["precision"], "recall": base["recall"], "f1": base["f1"],
            "f1_trivial": base["f1_all_positive"]}, preds

pooled_rows, POOLED_PREDS = [], []
for outcome in OUTCOMES:
    for fname in FEATURE_SETS:
        _collect = POOLED_PREDS
        r, _ = run_pooled(df_primary, outcome, feats_for(outcome, fname), CAT_COLS,
                          collect_preds=_collect, feature_label=fname,
                          z_method=Z_METHOD_PRIMARY)
        if r:
            r["features"] = fname
            pooled_rows.append(r)
    print(f"  {outcome} done")

pooled = pd.DataFrame(pooled_rows)
pooled_preds = (pd.concat(POOLED_PREDS, ignore_index=True) if POOLED_PREDS else pd.DataFrame())

print("\n" + "-" * 116)
print("Pooled model performance (averaged across cuts; 95% CI by participant bootstrap)")
print("-" * 116)
print(f"{'outcome':<12} {'features':<15} {'AUC (95% CI)':>20} {'AP':>6} {'chance':>7} "
      f"{'Prec':>6} {'Recall':>7} {'F1':>6} {'F1 triv':>8}")
print("-" * 116)

for _, r in pooled.iterrows():
    auc = (f"{r['auc']:.2f} ({r['auc_lo']:.2f}-{r['auc_hi']:.2f})"
           if pd.notna(r["auc_lo"]) else f"{r['auc']:.2f}")
    print(f"{r['outcome']:<12} {r['features']:<15} {auc:>20} {r['ap']:>6.2f} "
          f"{r['prevalence']:>7.2f} {r['precision']:>6.2f} {r['recall']:>7.2f} "
          f"{r['f1']:>6.2f} {r['f1_trivial']:>8.2f}")

print("\n" + "-" * 78)
print("INCREMENTAL VALUE OVER BASELINE (change in AUC)")
print("-" * 78)
note("\nRQ 1: do wearable domains add anything beyond baseline characteristics?"
     "Compare each change against width of the CIs above.")
for outcome, g in pooled.groupby("outcome"):
    b = g.loc[g["features"] == "baseline", "auc"]
    if b.empty: continue
    b = b.iloc[0]
    parts = [f"{fn.replace('bl_',''):<9}{g.loc[g['features']==fn,'auc'].iloc[0]-b:+.2f}"
             for fn in ["bl_activity","bl_physiol","bl_sleep","all"] if (g["features"]==fn).any()]
    print(f"  {outcome:<12} baseline={b:.2f}   " + "   ".join(parts))

RESULTS["pooled"] = pooled

# 7.1 - Benchmarks: is the signal between people or within them?
section("7.1 benchmarks: between-person vs within-person signal")

# 1. base-rate benchmark
bench_rows = []
for outcome in OUTCOMES:
    d = df_primary[df_primary[outcome].notna()]
    ys, ps = [], []
    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        tr, te = d[trm], d[~trm]
        if tr.empty or te.empty or te[outcome].nunique() < 2:
            continue
        rate = tr.groupby("story_id")[outcome].mean()
        ys.append(te[outcome].astype(int).to_numpy())
        ps.append(te["story_id"].map(rate).fillna(tr[outcome].mean()).to_numpy())
    if not ys:
        continue
    m = metrics(np.concatenate(ys), np.concatenate(ps))
    model_auc = pooled.loc[(pooled.outcome == outcome) & (pooled.features == "all"), "auc"]
    bench_rows.append({"outcome": outcome, "baserate_auc": m["auc"], "baserate_ap": m["ap"],
                       "model_auc": model_auc.iloc[0] if len(model_auc) else np.nan,
                       "prevalence": m["prevalence"]})

bench = pd.DataFrame(bench_rows)
bench["model_minus_baserate"] = bench["model_auc"] - bench["baserate_auc"]

print("-" * 78)
print("Base-rate benchmark")
print("-" * 78)
print(f"{'outcome':<10} {'base-rate AUC':>14} {'model AUC':>11} {'difference':>12}")
print("-" * 78)
for _, r in bench.iterrows():
    verdict = "model wins" if r["model_minus_baserate"] > 0 else "benchmark wins"
    print(f"{r['outcome']:<10} {r['baserate_auc']:>14.3f} {r['model_auc']:>11.3f} "
          f"{r['model_minus_baserate']:>+12.3f}   {verdict}")
print()
note("The benchmark uses no wearable data and predicts each person's own historical rate. "
     "If it wins, the pooled AUC is telling people apart rather than tracking change.")

# 2. within-person AUC
print("\n" + "-" * 78)
print("Within-person AUC")
print("-" * 78)

wp_rows = []
if not pooled_preds.empty:
    for (outcome,), g in pooled_preds.groupby(["outcome"]):
        aucs = []
        for pid, gp in g.groupby("story_id"):
            if gp["y"].nunique() == 2 and len(gp) >= 4:
                aucs.append(roc_auc_score(gp["y"], gp["proba"]))
        if not aucs:
            print(f"  {outcome:<10} no participants had both outcomes in test")
            continue
        a = np.array(aucs)
        wp_rows.append({"outcome": outcome, "n_participants": len(a),
                        "median_auc": np.median(a), "mean_auc": a.mean(),
                        "pct_above_chance": 100 * (a > 0.5).mean()})
        print(f"  {outcome:<10} n={len(a):>3}   median AUC={np.median(a):.3f}   "
              f"mean={a.mean():.3f}   above 0.5 in {100*(a>0.5).mean():.0f}% of people")
print()
note("A median near 0.5 means the model is not tracking change within individuals. "
     "That is compatible with a high pooled AUC, because the pooled figure is driven by differences BETWEEN people.")

RESULTS["baserate_benchmark"] = bench
RESULTS["within_person_auc"] = pd.DataFrame(wp_rows)

In [ ]:
# WEARABLE-ONLY: between-person vs within-person signal

w = pooled_preds[pooled_preds["features"] == "wearable_only"].copy()

for outcome, g in w.groupby("outcome"):
    # Between-person: correlation between each person's mean prediction
    # and their observed event rate in the test windows
    person = g.groupby("story_id").agg(
        observed_rate=("y", "mean"),
        predicted_rate=("proba", "mean")
    ).dropna()

    between_r = person["observed_rate"].corr(person["predicted_rate"], method="spearman")

    # Within-person: AUC after centring predictions within person
    aucs = []
    for pid, gp in g.groupby("story_id"):
        if gp["y"].nunique() == 2 and len(gp) >= 4:
            aucs.append(roc_auc_score(gp["y"], gp["proba"]))

    print(
        f"{outcome:<12} "
        f"between-person Spearman r={between_r:.2f}   "
        f"within-person median AUC={np.median(aucs):.3f} "
        f"(n={len(aucs)})"
    )

# 8. Person-specific models

- **Optimistic** - For each participant, best result across *3 cuts × 5 feature sets*. Signal detectable under optimal within-sample conditions?
- **Honest** - One pre-specified feature set (`all`), averaged across all cuts.

Hyperparameters are fixed here, not tuned, because few training windows per person.

In [ ]:
section("8. person-specific models: honest vs optimistic")

def run_person_specific(df, outcome, feature_sets, cat_cols, z_method="none"):
    """One model per participant: trained on their earlier windows, tested on
    their later ones. Hyperparameters are fixed (see fit_simple) because the
    training sets are far too small to tune on."""
    rows = []
    d = df[df[outcome].notna()].copy()
    for fname, feats in feature_sets.items():
        feats = [c for c in feats if c in d.columns]
        cats  = [c for c in feats if c in cat_cols]
        nums  = [c for c in feats if c not in cats]

        for frac in SPLIT_FRACTIONS:
            trm = person_cuts(d, frac)
            tr, te = d[trm], d[~trm]
            if tr.empty or te.empty:
                continue

            # same preprocessing as the pooled models
            _scale = None
            if z_method == "trainfold":
                _zc = zscoreable(nums)
                if _zc:
                    tr, te = apply_trainfold_z(tr, te, _zc)
                    _scale = [c for c in nums if c not in _zc]

            for pid, gte in te.groupby("story_id"):
                gtr = tr[tr["story_id"] == pid]
                if len(gtr) < MIN_TRAIN_WINDOWS:   continue
                if len(gte) < MIN_TEST_WINDOWS:    continue
                if gtr[outcome].nunique() < 2:     continue
                if gte[outcome].nunique() < 2:     continue
                m = fit_simple(gtr[feats], gtr[outcome], cats, nums, scale_cols=_scale)
                if m is None:                      continue
                p  = m.predict_proba(gte[feats])[:, 1]
                mt = metrics(gte[outcome], p)
                rows.append({"outcome": outcome, "features": fname, "frac": frac,
                             "z_method": z_method, "story_id": pid, "n_test": len(gte),
                             "auc": mt["auc"], "f1": mt["f1"], "ap": mt["ap"],
                             "f1_trivial": mt["f1_all_positive"]})
    return pd.DataFrame(rows)

person = pd.concat([run_person_specific(df_primary, o, sets_for(o), CAT_COLS,
                                       z_method=Z_METHOD_PERSON)
                    for o in OUTCOMES], ignore_index=True)

# pooled-model F1 per participant, matched on feature set
def _pooled_person_f1(pp_df, thresh=0.5):
    """F1 of the POOLED model on each participant's test windows, kept
    separate by feature set so comparison is like-for-like."""
    if pp_df.empty:
        return pd.DataFrame()
    has_f = "features" in pp_df.columns
    keys = ["outcome", "frac", "story_id"] + (["features"] if has_f else [])
    out = []
    for k, g in pp_df.groupby(keys):
        y  = g["y"].astype(int).to_numpy()
        pr = (g["proba"].to_numpy() >= thresh).astype(int)
        rec = dict(zip(keys, k if isinstance(k, tuple) else (k,)))
        rec["f1_pooled"] = f1_score(y, pr, zero_division=0)
        out.append(rec)
    return pd.DataFrame(out)

pooled_f1 = _pooled_person_f1(pooled_preds)
_iqr = lambda s: s.quantile(.75) - s.quantile(.25)
_HAS_FS = "features" in pooled_f1.columns

def _summarise_person(g, how, fname=None):
    cols = ["story_id", "auc", "f1", "ap", "f1_trivial", "n_test", "frac", "features"]
    if how == "honest":
        sub = g[g["features"] == fname]
        if sub.empty:
            return None
        per = (sub.groupby("story_id")
                  .agg(auc=("auc", "mean"), f1=("f1", "mean"), ap=("ap", "mean"),
                       f1_trivial=("f1_trivial", "mean"),
                       n_test=("n_test", "mean"), frac=("frac", "first")).reset_index())
        per["features"] = fname
        return per
    return (g.sort_values(["f1", "auc"], ascending=False)
             .groupby("story_id").head(1)[cols].reset_index(drop=True))

def _delta_vs_pooled(per, outcome, how):
    """Compare against the pooled model on the same windows and, where the
    information is available, the same feature set."""
    if pooled_f1.empty:
        return np.nan, np.nan
    pf = pooled_f1[pooled_f1["outcome"] == outcome]
    if pf.empty:
        return np.nan, np.nan
    join = ["story_id", "frac"] + (["features"] if _HAS_FS else [])
    if how == "honest":
        if _HAS_FS:
            pf = pf[pf["features"] == per["features"].iloc[0]]
            if pf.empty:                       # that set was not collected
                pf = pooled_f1[pooled_f1["outcome"] == outcome]
                join = ["story_id", "frac"]
        pf = pf.groupby("story_id")["f1_pooled"].mean().reset_index()
        m = per.merge(pf, on="story_id", how="left")
    else:
        m = per.merge(pf.drop(columns=["outcome"]), on=join, how="left")
    d = (m["f1"] - m["f1_pooled"]).dropna()
    return (np.nan, np.nan) if d.empty else (d.median(), 100 * (d >= 0.10).mean())

HDR = (f"{'outcome':<12}{'feature set':<15}{'N':>4}  {'AUC (IQR)':>14} "
       f"{'F1 (IQR)':>14} {'F1 triv':>9} {'AP (IQR)':>14} "
       f"{'dF1 vs pooled':>14} {'>=0.10':>8} {'test obs':>12}")

def _line(outcome, per, d_med, d_pct, show_fs):
    fs = per["features"].iloc[0] if show_fs else ""
    return (f"{outcome:<12}{fs:<15}{len(per):>4}  "
            f"{per['auc'].median():>6.2f} ({_iqr(per['auc']):>4.2f}) "
            f"{per['f1'].median():>6.2f} ({_iqr(per['f1']):>4.2f}) "
            f"{per['f1_trivial'].median():>9.2f} "
            f"{per['ap'].median():>6.2f} ({_iqr(per['ap']):>4.2f}) "
            f"{d_med:>+14.2f} {d_pct:>7.1f}% "
            f"{per['n_test'].median():>4.0f} ({per['n_test'].min():.0f}-{per['n_test'].max():.0f})")

print("-" * 132)
print("Person-specific model performance by outcome")
print("-" * 132)

if person.empty:
    print("  No person-specific models were estimable.")
    RESULTS["person_specific"] = pd.DataFrame()
else:
    table_rows = []

    print("\nEACH FEATURE SET, averaged across all cuts (no selection)")
    print(HDR); print("-" * 132)
    for outcome, g in person.groupby("outcome"):
        for fname in [f for f in FEATURE_SETS if f in set(g["features"])]:
            per = _summarise_person(g, "honest", fname)
            if per is None or per.empty:
                continue
            d_med, d_pct = _delta_vs_pooled(per, outcome, "honest")
            print(_line(outcome, per, d_med, d_pct, show_fs=True))
            table_rows.append({"reporting": "honest", "outcome": outcome,
                               "features": fname, "n_participants": len(per),
                               "median_auc": per["auc"].median(), "iqr_auc": _iqr(per["auc"]),
                               "median_f1": per["f1"].median(),   "iqr_f1": _iqr(per["f1"]),
                               "median_f1_trivial": per["f1_trivial"].median(),
                               "median_ap": per["ap"].median(),   "iqr_ap": _iqr(per["ap"]),
                               "median_delta_f1_vs_pooled": d_med,
                               "pct_ge_0.10_gain": d_pct,
                               "median_n_test": per["n_test"].median()})
        print()

    print("BEST CUT x FEATURE SET PER PARTICIPANT (as submitted)")
    print(HDR); print("-" * 132)
    for outcome, g in person.groupby("outcome"):
        per = _summarise_person(g, "optimistic")
        if per is None or per.empty:
            continue
        d_med, d_pct = _delta_vs_pooled(per, outcome, "optimistic")
        vc = per["features"].value_counts()
        sel = "   ".join(f"{fn} {int(vc.get(fn, 0))}/{len(per)}" for fn in FEATURE_SETS)
        print(_line(outcome, per, d_med, d_pct, show_fs=False))
        print(f"{'':<12}selected as best:  {sel}")
        table_rows.append({"reporting": "optimistic", "outcome": outcome,
                           "features": "best per participant", "n_participants": len(per),
                           "median_auc": per["auc"].median(), "iqr_auc": _iqr(per["auc"]),
                           "median_f1": per["f1"].median(),   "iqr_f1": _iqr(per["f1"]),
                           "median_f1_trivial": per["f1_trivial"].median(),
                           "median_ap": per["ap"].median(),   "iqr_ap": _iqr(per["ap"]),
                           "median_delta_f1_vs_pooled": d_med,
                           "pct_ge_0.10_gain": d_pct,
                           "median_n_test": per["n_test"].median(),
                           "most_selected": f"{vc.index[0]} ({vc.iloc[0]}/{len(per)})",
                           "selected_counts": sel})

    RESULTS["person_specific"] = pd.DataFrame(table_rows)
    print()

# 9. Deployment scenarios

Each participant's later windows are predicted three ways:

| Model | Trained on |
|---|---|
| A. Cohort only | other participants (new patient, no history) |
| B. Person only | this participant's earlier windows |
| C. Cohort + person | other participants plus this participant's earlier windows |

Cuts are reported separately to show how much personal history is needed before it helps. Participants are never in both the training and test data for model A.

In [ ]:
section("9. three-way validation")

def run_three_way(df, outcome, feats, cat_cols, n_folds=None):
    n_folds = THREE_WAY_N_COHORT_FOLDS if n_folds is None else n_folds
    feats = [c for c in feats if c in df.columns]
    cats  = [c for c in feats if c in cat_cols]
    nums  = [c for c in feats if c not in cats]
    d = df[df[outcome].notna()].copy()
    if d.empty or d[outcome].nunique() < 2:
        return pd.DataFrame()

    pids = d["story_id"].unique()
    rng  = np.random.default_rng(RANDOM_STATE)
    fold_of = {p: i % n_folds for i, p in enumerate(rng.permutation(pids))}
    d["_fold"] = d["story_id"].map(fold_of)

    rows = []
    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        # same preprocessing as sections 7 and 8, so model B is directly comparable
        _zc = zscoreable(nums) if Z_METHOD_PERSON == "trainfold" else []
        _scale = None
        if _zc:
            _tr_all, _te_all = apply_trainfold_z(d[trm], d[~trm], _zc)
            d_use = pd.concat([_tr_all, _te_all]).sort_index()
            _scale = [c for c in nums if c not in _zc]
        else:
            d_use = d
        for f in range(n_folds):
            cohort_tr = d_use[d_use["_fold"] != f]                     # other participants, all windows
            held      = d_use[d_use["_fold"] == f]
            if cohort_tr.empty or cohort_tr[outcome].nunique() < 2:
                continue
            # Model A is fitted once per fold - it never uses the held-out people
            mA = fit_simple(cohort_tr[feats], cohort_tr[outcome], cats, nums, scale_cols=_scale)
            for pid, g in held.groupby("story_id"):
                te = g[~trm.reindex(g.index, fill_value=False)]
                own = g[trm.reindex(g.index, fill_value=False)]
                if len(te) < MIN_TEST_WINDOWS or te[outcome].nunique() < 2:  continue
                if len(own) < MIN_TRAIN_WINDOWS:                  continue

                preds = {}
                if mA is not None:
                    preds["A_cohort"] = mA.predict_proba(te[feats])[:, 1]
                if own[outcome].nunique() >= 2:
                    mB = fit_simple(own[feats], own[outcome], cats, nums, scale_cols=_scale)
                    if mB is not None:
                        preds["B_person"] = mB.predict_proba(te[feats])[:, 1]
                both = pd.concat([cohort_tr, own], ignore_index=True)
                mC = fit_simple(both[feats], both[outcome], cats, nums, scale_cols=_scale)
                if mC is not None:
                    preds["C_cohort_plus"] = mC.predict_proba(te[feats])[:, 1]

                y = te[outcome].astype(int).to_numpy()
                for model, p in preds.items():
                    rows.append({"outcome": outcome, "frac": frac, "model": model,
                                 "story_id": pid, "n_test": len(te),
                                 "n_own_train": len(own),
                                 "auc": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan,
                                 "y": y, "proba": p})
    return pd.DataFrame(rows)

three = pd.concat([run_three_way(df_primary, o, feats_for(o, THREE_WAY_FEATURE_SET), CAT_COLS)
                   for o in OUTCOMES], ignore_index=True)

if three.empty:
    print("  No participants met the requirements for three-way validation.")
    RESULTS["three_way"] = pd.DataFrame()
else:
    LAB = {"A_cohort": "A cohort only", "B_person": "B person only *",
           "C_cohort_plus": "C cohort+person"}
    print("-" * 94)
    print("Per-person median AUC on identical test windows, by amount of personal history")
    print("-" * 94)
    rows = []
    for outcome, go in three.groupby("outcome"):
        print(f"\n  {outcome}")
        print(f"    {'model':<18}" + "".join(f"{int(f*100):>4}% cut " for f in SPLIT_FRACTIONS)
              + "   (n people at each cut)")
        for model in ["A_cohort", "B_person", "C_cohort_plus"]:
            gm = go[go["model"] == model]
            if gm.empty: continue
            cells, ns = [], []
            for frac in SPLIT_FRACTIONS:
                gf = gm[gm["frac"] == frac]
                if gf.empty:
                    cells.append("   -- "); ns.append(0); continue
                cells.append(f"{gf['auc'].median():>7.3f} "); ns.append(gf["story_id"].nunique())
                rows.append({"outcome": outcome, "model": model, "frac": frac,
                             "median_person_auc": gf["auc"].median(),
                             "n_people": gf["story_id"].nunique()})
            print(f"    {LAB[model]:<18}" + "".join(cells) + f"   ({', '.join(map(str,ns))})")
    RESULTS["three_way"] = pd.DataFrame(rows)
    print("\n" + "-" * 94)
    note("Read across a row for the dose-response in personal history; read down to "
         "compare the three deployment scenarios at a fixed amount of history.")

# 10. Who could be modelled individually

Person-specific models are only estimated for participants who have enough windows and vary across both outcomes.

We compare mean±SD for continuous and counts for categorical variables, plus standardised mean difference (exploratory, as N≈10), to see if those participants differ systematically from the rest.

**Reading the output**
* Last two rows: outcome base rate and within-person variability.
* These show how selected the modellable subgroup is.

In [ ]:
section("10. were the individually-modellable participants different?")

def smd(a, b):
    """Standardised mean difference (Cohen's d, pooled SD)."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2: return np.nan
    sp = np.sqrt(((len(a)-1)*a.var(ddof=1) + (len(b)-1)*b.var(ddof=1)) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan

def modellable_ids(df, outcome):
    """Participants for whom a person-specific model was estimable at any cut."""
    ok = set()
    d = df[df[outcome].notna()]
    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        tr, te = d[trm], d[~trm]
        for pid, gte in te.groupby("story_id"):
            gtr = tr[tr["story_id"] == pid]
            if (len(gtr) >= MIN_TRAIN_WINDOWS and len(gte) >= MIN_TEST_WINDOWS
                and gtr[outcome].nunique() >= 2 and gte[outcome].nunique() >= 2):
                ok.add(pid)
    return ok

CONT = [c for c in (BASE_NUM + ["n_valid_days", "frac_valid_days"]) if c in df_primary.columns]

for outcome in OUTCOMES:
    d = df_primary[df_primary[outcome].notna()]
    if d.empty: continue
    ids = modellable_ids(d, outcome)

    per = d.groupby("story_id").agg(
        n_windows=("date", "size"),
        base_rate=(outcome, "mean"),
        within_sd=(outcome, "std"),
        **{c: (c, "mean") for c in CONT})
    per["modellable"] = per.index.isin(ids)
    g1, g0 = per[per["modellable"]], per[~per["modellable"]]
    if len(g1) < 2 or len(g0) < 2:
        print(f"\n  {outcome}: {len(g1)} modellable of {len(per)} - too few to compare")
        continue

    print(f"\n  {outcome}: {len(g1)} of {len(per)} participants modellable individually "
          f"({100*len(g1)/len(per):.0f}%)")
    print(f"    {'variable':<24}{'modellable':>18}{'not modellable':>18}{'std. diff':>11}")
    print("    " + "-" * 71)
    for v in ["n_windows"] + CONT + ["base_rate", "within_sd"]:
        if v not in per.columns: continue
        a, b = g1[v], g0[v]
        print(f"    {v:<24}{a.mean():>10.2f} ({a.std():>4.2f}){b.mean():>10.2f} ({b.std():>4.2f})"
              f"{smd(a, b):>11.2f}")

    for cvar in [c for c in ["ed_type", "treatment_status_screening"] if c in df_primary.columns]:
        lv = df_primary.drop_duplicates("story_id").set_index("story_id")[cvar]
        print(f"    {cvar}:")
        for level in lv.dropna().unique():
            n1 = int((lv.reindex(g1.index) == level).sum())
            n0 = int((lv.reindex(g0.index) == level).sum())
            print(f"       {str(level):<21}{n1:>10} ({100*n1/max(1,len(g1)):>3.0f}%)"
                  f"{n0:>10} ({100*n0/max(1,len(g0)):>3.0f}%)")

print()
note("Standardised difference >0.5 moderate difference, >0.8 large.")

# 11. Feature importance (exploratory)

Permutation importance: shuffle one feature, see how much AUC drops. 

**Caveats**:
1. Elastic net shrinks most coefficients to zero at this sample size, so shuffling those features changes nothing. Ranking them is meaningless, report how many features tie at zero.
2. **Whether the top features are time-invariant.** If `bmi_baseline`, `age_numeric`,
   `illness_duration_years` or diagnosis dominate, the model is doing between-person discrimination.

In [ ]:
section("11. exploratory permutation feature importance")

def feature_domain(col):
    if col in BASE_NUM or col in BASE_CAT:
        return "baseline"
    for dom, bases in (("activity", ACTIVITY_BASES),
                       ("physiology", PHYSIO_BASES),
                       ("sleep", SLEEP_BASES)):
        for b in bases:
            if col.startswith(b + "_"):
                return dom
    return "other"

def domain_shares(feats, imp):
    """Percentage of total importance carried by each domain. Negative values
    (shuffling helped) are set to zero before sharing out."""
    d = pd.DataFrame({"feature": list(feats), "imp": np.asarray(imp, float)})
    d["imp"] = d["imp"].clip(lower=0)
    d["domain"] = d["feature"].map(feature_domain)
    tot = d["imp"].sum()
    if tot <= 0:
        return pd.Series(dtype=float)
    return 100 * d.groupby("domain")["imp"].sum() / tot

# pooled: computed at EVERY cut-off, then averaged
imp_rows, dom_rows, imp_summary = [], [], []
TIME_INVARIANT = set(BASE_NUM + BASE_CAT)

for outcome in OUTCOMES:
    d = df_primary[df_primary[outcome].notna()]
    feats = [c for c in feats_for(outcome, "all") if c in d.columns]
    cats  = [c for c in feats if c in CAT_COLS]
    nums  = [c for c in feats if c not in cats]

    per_cut = []                       # one importance vector per cut-off
    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        tr, te = d[trm], d[~trm]
        if tr.empty or te.empty or te[outcome].nunique() < 2:
            continue
        _zc = zscoreable(nums) if Z_METHOD_PRIMARY == "trainfold" else []
        _scale = None
        if _zc:
            tr, te = apply_trainfold_z(tr, te, _zc)
            _scale = [c for c in nums if c not in _zc]
        model = fit_tuned(tr[feats], tr[outcome], tr["story_id"], tr["date"],
                          cats, nums, scale_cols=_scale)
        if model is None:
            continue
        pi = permutation_importance(model, te[feats], te[outcome].astype(int),
                                    scoring="roc_auc", n_repeats=N_PERM_REPEATS,
                                    random_state=RANDOM_STATE, n_jobs=-1)
        per_cut.append(pi.importances_mean)

    if not per_cut:
        continue
    m  = np.nanmean(np.vstack(per_cut), axis=0)      # averaged across cut-offs
    sd = np.nanstd(np.vstack(per_cut), axis=0)       # spread ACROSS cut-offs
    n_zero = int(np.sum(np.abs(m) < 1e-12))

    sh = domain_shares(feats, m)
    for dom, pct in sh.items():
        dom_rows.append({"level": "pooled", "outcome": outcome, "domain": dom, "pct": pct})

    ranked = sorted(zip(feats, m, sd), key=lambda t: -t[1])
    top = [t for t in ranked if abs(t[1]) >= 1e-12][:8]
    n_ti = sum(1 for f, _, _ in top[:5] if f in TIME_INVARIANT)
    print(f"\n  {outcome}: {len(feats)} features, averaged over {len(per_cut)} cut-offs | "
          f"{n_zero} tie at exactly zero ({100*n_zero/len(feats):.0f}%) | "
          f"{n_ti}/5 of the top 5 are time-invariant")
    print("     domain share: " + ", ".join(f"{k} {v:.1f}%" for k, v in
                                            sh.sort_values(ascending=False).items()))
    for f, mu, s_ in top:
        print(f"     {mu:+.4f} (SD across cut-offs {s_:.4f})  {f}"
              + ("  [time-invariant]" if f in TIME_INVARIANT else ""))
        imp_rows.append({"outcome": outcome, "feature": f, "auc_drop": mu, "sd_across_cuts": s_,
                         "domain": feature_domain(f), "time_invariant": f in TIME_INVARIANT})
    imp_summary.append({"outcome": outcome, "n_features": len(feats), "n_cuts": len(per_cut),
                        "n_zero": n_zero, "pct_zero": 100*n_zero/len(feats),
                        "n_time_invariant_in_top5": n_ti})

# person-specific: also averaged across cut-offs
print("\n" + "-" * 78)
print("PERSON-SPECIFIC IMPORTANCE - domain share within each participant")
print("-" * 78)
note("Exploratory. Each participant's own model permuted on their held-out windows, "
     "based on 5-15 observations, thus unstable. Averaged across cut-offs, then "
     "reported as medians across participants by domain.")

ps_raw, ps_top = [], []
for outcome in OUTCOMES:
    d = df_primary[df_primary[outcome].notna()]
    feats = [c for c in feats_for(outcome, "all") if c in d.columns]
    cats  = [c for c in feats if c in CAT_COLS]
    nums  = [c for c in feats if c not in cats]

    for frac in SPLIT_FRACTIONS:
        trm = person_cuts(d, frac)
        tr, te = d[trm], d[~trm]
        if tr.empty or te.empty:
            continue
        _zc = zscoreable(nums) if Z_METHOD_PERSON == "trainfold" else []
        _scale = None
        if _zc:
            tr, te = apply_trainfold_z(tr, te, _zc)
            _scale = [c for c in nums if c not in _zc]

        for pid, gte in te.groupby("story_id"):
            gtr = tr[tr["story_id"] == pid]
            if len(gtr) < MIN_TRAIN_WINDOWS or len(gte) < MIN_TEST_WINDOWS:  continue
            if gtr[outcome].nunique() < 2 or gte[outcome].nunique() < 2:     continue
            mdl = fit_simple(gtr[feats], gtr[outcome], cats, nums, scale_cols=_scale)
            if mdl is None:                                                  continue
            pi = permutation_importance(mdl, gte[feats], gte[outcome].astype(int),
                                        scoring="roc_auc", n_repeats=N_PERM_REPEATS,
                                        random_state=RANDOM_STATE, n_jobs=-1)
            sh = domain_shares(feats, pi.importances_mean)
            for dom, pct in sh.items():
                ps_raw.append({"outcome": outcome, "story_id": pid,
                               "frac": frac, "domain": dom, "pct": pct})

            # top-10 features for this participant at this cut, for the
            # selection-frequency (stability) summary
            _order = np.argsort(-pi.importances_mean)
            for j in _order[:10]:
                if pi.importances_mean[j] > 1e-12:
                    ps_top.append({"outcome": outcome, "story_id": pid,
                                   "frac": frac, "feature": feats[j]})

if ps_raw:
    # average each participant across their cut-offs first, then summarise
    psd = (pd.DataFrame(ps_raw)
             .groupby(["outcome", "story_id", "domain"])["pct"].mean().reset_index())
    for outcome, g in psd.groupby("outcome"):
        n = g["story_id"].nunique()
        agg = (g.groupby("domain")["pct"]
                 .agg(median="median", q25=lambda x: x.quantile(.25),
                      q75=lambda x: x.quantile(.75))
                 .sort_values("median", ascending=False))
        print(f"\n  {outcome} (n={n} participants)")
        for dom, r in agg.iterrows():
            print(f"     {dom:<12} {r['median']:>5.1f}%  (IQR {r['q25']:.1f}-{r['q75']:.1f})")
        for dom, r in agg.iterrows():
            dom_rows.append({"level": "person", "outcome": outcome, "domain": dom,
                             "pct": r["median"], "q25": r["q25"], "q75": r["q75"],
                             "n_participants": n})
else:
    print("  No person-specific models were estimable.")

# selection frequency across participants
if ps_top:
    print("\n" + "-" * 78)
    print("SELECTION FREQUENCY - how often a feature reached a participant's top 10")
    print("-" * 78)
    note("Counted once per participant, not once per cut-off. A flat distribution, with "
         "no feature appearing for many participants, indicates that individual feature "
         "rankings are unstable and should not be interpreted (Nogueira et al., 2018).")
    _t = pd.DataFrame(ps_top).drop_duplicates(["outcome", "story_id", "feature"])
    for outcome, g in _t.groupby("outcome"):
        n = g["story_id"].nunique()
        vc = g["feature"].value_counts().head(8)
        print(f"\n  {outcome} (n={n} participants)")
        for f, k in vc.items():
            print(f"     {k:>2}/{n} ({100*k/n:>3.0f}%)  {f}")
            dom_rows.append({"level": "person_topN", "outcome": outcome, "feature": f,
                             "n_participants_selecting": int(k), "n_participants": n,
                             "pct": 100 * k / n})

RESULTS["importance"] = pd.DataFrame(imp_rows)
RESULTS["importance_summary"] = pd.DataFrame(imp_summary)
RESULTS["importance_domains"] = pd.DataFrame(dom_rows)

# 12. Sensitivity checks

1. **Wear-time threshold** - how many windows survive
2. **Minimum ED-15 completions** - 3 to 10, with bootstrap CIs
3. **Window definition** - `full14` vs `antecedent` vs `full14_next`

All three run on the `all` feature set only, using the same within-person centring as the
primary analysis (`Z_METHOD_PRIMARY`). Check 2 bootstraps `N_BOOTSTRAP_Z` times; checks 1 and 3
report point estimates only, to keep runtime sensible.

In [ ]:
section("12. sensitivity checks")

# 12.1 wear-time threshold
print("\n12.1 - Wear-time threshold\n")
print(f"{'max non-wear (s)':>18} {'min wear (h)':>13} {'valid days %':>13} {'windows kept':>14}")
print("-" * 62)
for thr in [15_000, 25_000, 35_000]:
    tmp = oura_daily.copy()
    tmp["day_valid"] = (pd.to_numeric(tmp["non_wear_time"], errors="coerce") < thr).fillna(False)
    dd = build_windows(tmp, ed15_el, PRIMARY_WINDOW, verbose=False)
    print(f"{thr:>18,} {24-thr/3600:>13.1f} {100*tmp['day_valid'].mean():>12.1f}% {len(dd):>14,}")

# 12.2 minimum ED-15 completions
print("\n12.2 - Minimum ED-15 completions\n")
note("No precedent for completion threshold. Read 95%CI width column.")
print(f"\n{'min completions':>16} {'participants':>13} {'windows':>9} "
      f"{'Restrict AUC (95% CI)':>24} {'CI width':>9}")
print("-" * 78)
_saved = MIN_N_ED15
mc_rows = []
for mn in range(3, 11):
    globals()["MIN_N_ED15"] = mn
    cnt = ed15.groupby("story_id").size()
    sub = ed15[ed15["story_id"].isin(cnt[cnt >= mn].index)].copy()
    dd = build_windows(oura_daily, sub, PRIMARY_WINDOW, verbose=False)
    dd, _ = attach_baseline(dd)
    if dd.empty:
        print(f"{mn:>16} {'--':>13} {'--':>9} {'--':>24} {'--':>9}"); continue
    fs, cc = build_feature_sets(dd)
    r, _ = run_pooled(dd, "Restrict", fs["all"], cc,
                      n_boot=N_BOOTSTRAP_Z, z_method=Z_METHOD_PRIMARY)
    if not r:
        print(f"{mn:>16} {dd['story_id'].nunique():>13} {len(dd):>9,} {'--':>24} {'--':>9}")
        continue
    wid = (r["auc_hi"] - r["auc_lo"]) if pd.notna(r.get("auc_lo")) else np.nan
    cell = (f"{r['auc']:.2f} ({r['auc_lo']:.2f}-{r['auc_hi']:.2f})"
            if pd.notna(r.get("auc_lo")) else f"{r['auc']:.2f}")
    print(f"{mn:>16} {dd['story_id'].nunique():>13} {len(dd):>9,} {cell:>24} {wid:>9.2f}")
    mc_rows.append({"min_completions": mn, "n_people": dd["story_id"].nunique(),
                    "n_windows": len(dd), "auc": r["auc"],
                    "auc_lo": r["auc_lo"], "auc_hi": r["auc_hi"], "ci_width": wid})
globals()["MIN_N_ED15"] = _saved
RESULTS["min_completions"] = pd.DataFrame(mc_rows)

# decision rule
if mc_rows:
    _m = pd.DataFrame(mc_rows)
    _best = _m.loc[_m["ci_width"].idxmin()]
    _lo3, _hi3 = _m.iloc[0]["auc_lo"], _m.iloc[0]["auc_hi"]
    _sep = (_m["auc"] > _hi3) | (_m["auc"] < _lo3)
    print("-" * 78)
    print(f"  Narrowest interval: min = {int(_best['min_completions'])} "
          f"(width {_best['ci_width']:.2f}, n = {int(_best['n_people'])} participants)")
    if _sep.any():
        print(f"  Thresholds whose AUC falls outside the min=3 interval: "
              f"{', '.join(str(int(v)) for v in _m.loc[_sep, 'min_completions'])}")
        print("  -> short series do carry less signal; the threshold is a real choice.")
    else:
        print("  Every AUC sits inside the min=3 interval.")
        print("  -> the threshold does not materially change the estimate. Take the")
        print("     value that keeps the most participants, and report the rest here.")

# 12.3 window definition
def _within_person_auc(pred_list):
    """Median AUC computed separately within each participant, as in section 7.1."""
    if not pred_list:
        return np.nan, 0
    pr = pd.concat(pred_list, ignore_index=True)
    a = [roc_auc_score(g["y"], g["proba"]) for _pid, g in pr.groupby("story_id")
         if g["y"].nunique() == 2 and len(g) >= 4]
    return (float(np.median(a)), len(a)) if a else (np.nan, 0)

print("\n12.3 - Window definition\n")
print(f"{'window':<12} {'days':>10} {'participants':>13} {'windows':>9}  " +
      "  ".join(f"{o+' AUC':>13}" for o in OUTCOMES))
print("-" * 92)
sens_rows, wp_cells = [], {}
for w, d in datasets.items():
    if d.empty: continue
    fs, cc = build_feature_sets(d)
    _ex = [x for x in CHANGE_COVARIATES if x in d.columns]
    aucs = {}
    for o in OUTCOMES:
        # Deteriorate carries the previous score and gap, as in section 5
        _feats = fs["all"] + (_ex if o == "Deteriorate" else [])
        _pred = []
        r, _ = run_pooled(d, o, _feats, cc, n_boot=0, z_method=Z_METHOD_PRIMARY,
                          collect_preds=_pred, feature_label="all")
        aucs[o] = r["auc"] if r else np.nan
        wp_cells[(w, o)] = _within_person_auc(_pred)
    # full14_next is derived from full14 after the loop, so it has no WINDOWS entry:
    # same sensor days, outcome taken from the following completion.
    ds, de, _ = WINDOWS.get(w, WINDOWS["full14"])
    _span = f"-{ds} to -{de}" + (" *" if w not in WINDOWS else "")
    print(f"{w:<12} {_span:>12} {d['story_id'].nunique():>13} {len(d):>9,}  " +
          "  ".join(f"{aucs[o]:>13.3f}" for o in OUTCOMES))
    sens_rows.append({"window": w, **{f"{o}_auc": aucs[o] for o in OUTCOMES},
                      **{f"{o}_within_auc": wp_cells[(w, o)][0] for o in OUTCOMES},
                      **{f"{o}_within_n": wp_cells[(w, o)][1] for o in OUTCOMES}})

print("\n  Same models, scored within each participant instead of across the sample")
print(f"{'window':<12} " + "  ".join(f"{o:>18}" for o in OUTCOMES))
print("-" * 92)
for row in sens_rows:
    w = row["window"]
    print(f"{w:<12} " + "  ".join(
        f"{row[o+'_within_auc']:>12.3f} ({row[o+'_within_n']:>3})" if pd.notna(row[o+"_within_auc"])
        else f"{'--':>18}" for o in OUTCOMES))
print("\n  Median of each participant's own AUC; n = participants with both outcome")
print("  states and at least 4 test windows, so each cell rests on a different subset.")

print("\n  * full14_next uses the same sensor days as full14; the outcome is taken")
print("    from the FOLLOWING completion, so the two rows are not the same analysis.")
RESULTS["sensitivity_windows"] = pd.DataFrame(sens_rows)

# 13. Figures and result files

In [ ]:
section("13. figures")

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})
FIGDIR = OUT_DIR / "figures"; FIGDIR.mkdir(parents=True, exist_ok=True)
_FS = ["baseline", "bl_activity", "bl_physiol", "bl_sleep", "all", "wearable_only"]
FS_LABEL = {"baseline":    "Baseline",
            "bl_activity": "Baseline + activity",
            "bl_physiol":  "Baseline + physiology",
            "bl_sleep":    "Baseline + sleep",
            "all":         "Baseline + all domains",
            "wearable_only": "Wearable only"}
OUTCOME_LABEL = {
    "Deteriorate": "Total score"
}

_rng = np.random.default_rng(RANDOM_STATE)
_outs = [o for o in OUTCOMES if o in set(RESULTS["pooled"]["outcome"])]

# ROC, best-performing pooled model per outcome
_best = RESULTS["pooled"].loc[RESULTS["pooled"].groupby("outcome")["auc"].idxmax()] \
                         .set_index("outcome")
fig, ax = plt.subplots(figsize=(4.6, 4.4))
for c, o in zip(["#2E5496", "#C0392B", "#1E8449", "#B7950B"], _outs):
    if o not in _best.index:
        continue
    fname = _best.loc[o, "features"]
    g = pooled_preds[(pooled_preds.outcome == o) & (pooled_preds.features == fname)]
    if g.empty or g["y"].nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(g["y"], g["proba"])
    ax.plot(fpr, tpr, color=c, lw=1.6,
            label=f"{OUTCOME_LABEL.get(o, o)} — {FS_LABEL.get(fname, fname)} (AUC {_best.loc[o,'auc']:.2f})")
ax.plot([0, 1], [0, 1], color="#999", ls=":", lw=1)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
ax.legend(loc="lower right", fontsize=7.5, frameon=False)
ax.set_title("ROC, best-performing pooled model per outcome", fontsize=9)
fig.tight_layout(); fig.savefig(FIGDIR / "fig1_roc.png", dpi=300)

# pooled AUC with CIs, against the base-rate benchmark
_p, _b = RESULTS["pooled"], RESULTS.get("baserate_benchmark", pd.DataFrame())
_outs = [o for o in OUTCOMES if o in set(_p["outcome"])]
fig, axes = plt.subplots(1, len(_outs), figsize=(2.8 * len(_outs), 3.6), sharex=True)
for ax, o in zip(np.atleast_1d(axes), _outs):
    g = _p[_p.outcome == o].set_index("features").reindex(_FS).dropna(subset=["auc"])
    y = np.arange(len(g))[::-1]
    ax.errorbar(g["auc"], y, xerr=[g["auc"] - g["auc_lo"], g["auc_hi"] - g["auc"]],
                fmt="o", ms=4, lw=1.2, color="#2E5496", capsize=2)
    ax.axvline(.5, color="#999", ls=":", lw=1)
    if not _b.empty and (_b.outcome == o).any():
        ax.axvline(_b.loc[_b.outcome == o, "baserate_auc"].iloc[0],
                   color="#C0392B", ls="--", lw=1.2)
        ax.set_yticks(y)
        ax.set_yticklabels([FS_LABEL.get(f, f) for f in g.index] if o == _outs[0] else [])
        ax.set_title(OUTCOME_LABEL.get(o, o), fontsize=10); ax.set_xlim(.3, 1.0); ax.set_xlabel("AUC")
fig.suptitle("Pooled discrimination (95% CI). Dotted = chance; dashed red = base-rate benchmark",
             fontsize=9, y=1.02)
fig.tight_layout(); fig.savefig(FIGDIR / "fig2_pooled_forest.png", dpi=300, bbox_inches="tight")

# within-person AUC
_rows = []
if not pooled_preds.empty:
    for o, g in pooled_preds.groupby("outcome"):
        for pid, gp in g.groupby("story_id"):
            if gp["y"].nunique() == 2 and len(gp) >= 4:
                _rows.append({"outcome": o, "auc": roc_auc_score(gp["y"], gp["proba"])})
_wp = pd.DataFrame(_rows)
if not _wp.empty:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for i, o in enumerate(_outs):
        v = _wp.loc[_wp.outcome == o, "auc"].to_numpy()
        if not len(v): continue
        ax.scatter(np.full(len(v), i) + _rng.normal(0, .06, len(v)), v,
                   s=16, alpha=.55, color="#2E5496")
        ax.plot([i - .25, i + .25], [np.median(v)] * 2, color="#C0392B", lw=2)
    ax.axhline(.5, color="#999", ls=":", lw=1)
    ax.set_xticks(range(len(_outs))); ax.set_xticklabels([OUTCOME_LABEL.get(o, o) for o in _outs])
    ax.set_ylabel("Within-person AUC"); ax.set_ylim(-.02, 1.02)
    ax.set_title("Discrimination within each participant (red = median, dotted = chance)",
                 fontsize=9)
    fig.tight_layout(); fig.savefig(FIGDIR / "fig3_within_person.png", dpi=300)

_opt = (person.sort_values(["f1", "auc"], ascending=False)
              .groupby(["outcome", "story_id"]).head(1))

if not _wp.empty:
    fig, ax = plt.subplots(figsize=(6, 3.4))
    for i, o in enumerate(_outs):
        for dx, col, src, lab in [(-0.15, "#2E5496", _wp.loc[_wp.outcome == o, "auc"], "pooled model"),
                                  (+0.15, "#1E8449", _opt.loc[_opt.outcome == o, "auc"], "person-specific (best cut x set)")]:
            v = src.to_numpy()
            if not len(v): continue
            ax.scatter(np.full(len(v), i + dx) + _rng.normal(0, .04, len(v)), v,
                       s=16, alpha=.55, color=col,
                       label=lab if i == 0 else None)
            ax.plot([i + dx - .11, i + dx + .11], [np.median(v)] * 2, color="#C0392B", lw=2)
    ax.axhline(.5, color="#999", ls=":", lw=1)
    ax.set_xticks(range(len(_outs))); ax.set_xticklabels([OUTCOME_LABEL.get(o, o) for o in _outs])
    ax.set_ylabel("AUC within each participant"); ax.set_ylim(-.02, 1.02)
    ax.legend(fontsize=7.5, frameon=False, loc="lower right")
    ax.set_title("Discrimination within each participant (red = median, dotted = chance)",
                 fontsize=9)
    fig.tight_layout(); fig.savefig(FIGDIR / "fig3_within_person.png", dpi=300)

# honest vs optimistic
if not person.empty:
    fig, axes = plt.subplots(1, len(_outs), figsize=(2.8 * len(_outs), 3.4), sharey=True)
    for ax, o in zip(np.atleast_1d(axes), _outs):
        g   = person[person.outcome == o]
        hon = g[g.features == "all"].groupby("story_id")["auc"].mean()
        opt = (g.sort_values(["f1", "auc"], ascending=False)
                 .groupby("story_id").head(1).set_index("story_id")["auc"])
        both = pd.concat([hon.rename("honest"), opt.rename("optimistic")], axis=1).dropna()
        for _, r in both.iterrows():
            ax.plot([0, 1], [r["honest"], r["optimistic"]], color="#BBB", lw=.8, zorder=1)
        ax.scatter(np.zeros(len(both)), both["honest"],     s=18, color="#2E5496", zorder=2)
        ax.scatter(np.ones(len(both)),  both["optimistic"], s=18, color="#C0392B", zorder=2)
        ax.axhline(.5, color="#999", ls=":", lw=1)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["honest", "optimistic"])
        ax.set_xlim(-.35, 1.35); ax.set_title(f"{OUTCOME_LABEL.get(o, o)} (n={len(both)})", fontsize=10)
    axes[0].set_ylabel("Person-specific AUC")
    fig.suptitle("Effect of selecting the best cut and feature set per participant",
                 fontsize=9, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig4_honest_vs_optimistic.png", dpi=300, bbox_inches="tight")

# three deployment scenarios
_LAB = {"A_cohort": "A cohort only", "B_person": "B person only",
        "C_cohort_plus": "C cohort + person"}
_CLR = {"A_cohort": "#888888", "B_person": "#C0392B", "C_cohort_plus": "#2E5496"}
_LS  = {"A_cohort": "-", "B_person": "-", "C_cohort_plus": "-"}
_MK  = {"A_cohort": "s", "B_person": "^", "C_cohort_plus": "o"}
_Z   = {"A_cohort": 2, "B_person": 3, "C_cohort_plus": 3}
_OFF = {"A_cohort": -0.6, "B_person": 0.0, "C_cohort_plus": 0.6}  # % pts, x-jitter

_t = RESULTS.get("three_way", pd.DataFrame())
if not _t.empty:
    _to = [o for o in OUTCOMES if o in set(_t.outcome)]
    fig, axes = plt.subplots(1, len(_to), figsize=(2.8 * len(_to), 3.5), sharey=True)
    for ax, o in zip(np.atleast_1d(axes), _to):
        g = _t[_t.outcome == o]
        for m in ["A_cohort", "B_person", "C_cohort_plus"]:
            gm = g[g.model == m].sort_values("frac")
            if gm.empty:
                continue
            ax.plot(gm["frac"] * 100 + _OFF[m], gm["median_person_auc"],
                     marker=_MK[m], ms=4, lw=1.5, ls=_LS[m],
                     color=_CLR[m], zorder=_Z[m],
                     label=_LAB[m] if o == _to[0] else None)
        ax.axhline(.5, color="#999", ls=":", lw=1)
        ax.set_title(OUTCOME_LABEL.get(o, o), fontsize=10)
        ax.set_xlabel("% of timeline used for training")
        ax.set_xticks([int(f * 100) for f in SPLIT_FRACTIONS]); ax.set_ylim(0.0, 1.0)
    axes[0].set_ylabel("Per-person median AUC on identical test windows")
    fig.legend(loc="lower center", ncol=3, frameon=False, fontsize=8,
               bbox_to_anchor=(.5, -.06))
    fig.suptitle("Does prediction improve as personal history accumulates?",
                 fontsize=9, y=1.02)
    fig.tight_layout()
    fig.savefig(FIGDIR / "fig5_deployment_scenarios.png", dpi=300, bbox_inches="tight")

print(f"  figures written to {FIGDIR}")

# pooled vs person-specific, one point per participant
_opt = (person.sort_values(["f1", "auc"], ascending=False)
              .groupby(["outcome", "story_id"]).head(1))
_cmp = (_opt.merge(pooled_f1, on=["outcome", "frac", "story_id", "features"], how="left")
            .dropna(subset=["f1_pooled"]))

_po = [o for o in OUTCOMES if o in set(_cmp.outcome)]
fig, axes = plt.subplots(1, len(_po), figsize=(2.6 * len(_po), 3.2),
                         sharex=True, sharey=True)
for ax, o in zip(np.atleast_1d(axes), _po):
    g = _cmp[_cmp.outcome == o]
    gain = g["f1"] - g["f1_pooled"]
    ax.scatter(g["f1_pooled"], g["f1"],
           s=22, alpha=.65, color="#888888", zorder=2)
    ax.plot([0, 1], [0, 1], color="#444", ls="--", lw=1, zorder=1)
    ax.set_xlim(-.02, 1.02); ax.set_ylim(-.02, 1.02); ax.set_aspect("equal")
    ax.set_title(f"{OUTCOME_LABEL.get(o, o)}\nn={len(g)}", fontsize=9)
    ax.set_xlabel("Pooled model F1")
axes[0].set_ylabel("Person-specific F1")
fig.suptitle("Pooled versus person-specific performance, by participant\n",
             fontsize=9, y=1.06)
fig.tight_layout()
fig.savefig(FIGDIR / "fig6_pooled_vs_person.png", dpi=300, bbox_inches="tight")

In [ ]:
# Save result tables
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, obj in RESULTS.items():
    if isinstance(obj, pd.DataFrame) and not obj.empty:
        obj.to_csv(OUT_DIR / f"result_{name}.csv", index=False)
if not pooled_preds.empty:
    pooled_preds.to_csv(OUT_DIR / "predictions_pooled.csv", index=False)
print(f"\nSaved result tables to: {OUT_DIR}")

# Appendix A - Building `oura_daily_clean.csv`

Run this to (re-)generate the Oura daily file.

*Note:* Fixes sleep inflation by de-duplicating sleep folders on `(story_id, day, bedtime_start)` before being summed.

In [ ]:
import os, re

def rebuild_oura_from_raw():
    section("Appendix A - rebuilding the daily Ōura file from raw server data")

    ids_df = pd.read_csv(COHORT_CSV, low_memory=False)

    def _yes(col):
        if col not in ids_df.columns:
            print(f"   !! column '{col}' not found, ignoring"); return pd.Series(False, index=ids_df.index)
        return ids_df[col].astype(str).str.strip().str.lower().isin(["1","1.0","true","yes","y","t"])

    keep = _yes("is_valid") & _yes("has_oura") & ~_yes("flag_test_account") & ~_yes("flag_fraud")
    story_ids = set(ids_df.loc[keep, "story_id"].dropna().astype(int).astype(str))
    print(f"  Eligible participants: {len(story_ids)}")

    # map Ōura account name -> story_id
    norm = lambda x: re.sub(r"[^a-z0-9]", "", str(x).strip().lower())
    src  = ids_df.loc[keep, ["story_id", "oura_id"]].dropna(subset=["oura_id"])
    by_name, digits = {}, {}
    for _, r in src.iterrows():
        sid, raw = int(r["story_id"]), str(r["oura_id"])
        if norm(raw): by_name[norm(raw)] = sid
        d = re.sub(r"\D", "", raw)
        if d: digits.setdefault(d, set()).add(sid)
    by_digit = {k: next(iter(v)) for k, v in digits.items() if len(v) == 1}
    ambiguous = {k: v for k, v in digits.items() if len(v) > 1}
    if ambiguous:
        print(f"  !! {len(ambiguous)} digit strings map to >1 participant, excluded from fallback")

    def to_sid(nm):
        n = norm(nm)
        if n in by_name: return by_name[n]
        d = re.sub(r"\D", "", str(nm))
        return by_digit.get(d)

    def load_folder(folder, cols, rename=None, label=""):
        recs, n_files = [], 0
        for root, _dirs, files in os.walk(folder):
            for fn in sorted(files):
                if not fn.lower().endswith(".csv"): continue
                path = os.path.join(root, fn)
                try:    head = pd.read_csv(path, nrows=1)
                except Exception: continue
                if "name" not in head.columns: continue
                sid = to_sid(head["name"].iloc[0]) if len(head) else None
                if sid is not None and str(sid) not in story_ids: continue
                try:    d = pd.read_csv(path, low_memory=False)
                except Exception: continue
                if d.empty or "name" not in d.columns: continue
                d["story_id"] = d["name"].map(to_sid)
                d = d[d["story_id"].notna()]
                if d.empty: continue
                d["story_id"] = d["story_id"].astype(int).astype(str)
                d = d[d["story_id"].isin(story_ids)]
                missing = [c for c in cols if c not in d.columns]
                if d.empty or missing: continue
                d = d[cols].rename(columns=rename or {})
                recs.append(d); n_files += 1
        if not recs:
            print(f"  {label:<12} no usable files"); return pd.DataFrame()
        out = pd.concat(recs, ignore_index=True)
        n0 = len(out); out = out.drop_duplicates()
        if "day" in out.columns:
            dd = pd.to_datetime(out["day"], errors="coerce")
            out = out[(dd <= DATA_CUTOFF_TS) | dd.isna()]
        print(f"  {label:<12} {n_files:>4} files | {n0:>7,} -> {len(out):>7,} rows "
              f"| {out['story_id'].nunique()} participants")
        return out

    A = load_folder(str(OURA_RAW_DIR / "dailyactivity"),
        ["story_id","name","day","score","steps","equivalent_walking_distance","total_calories",
         "active_calories","average_met_minutes","high_activity_met_minutes","high_activity_time",
         "low_activity_met_minutes","low_activity_time","medium_activity_met_minutes",
         "medium_activity_time","non_wear_time","resting_time","sedentary_met_minutes",
         "sedentary_time"], {"score":"activity_score"}, "activity")
    DS = load_folder(str(OURA_RAW_DIR / "dailysleep"), ["story_id","day","score"],
                     {"score":"sleep_score"}, "dailysleep")
    SL = load_folder(str(OURA_RAW_DIR / "sleep"),
        ["story_id","day","type","efficiency","total_sleep_duration","light_sleep_duration",
         "deep_sleep_duration","rem_sleep_duration","average_breath","average_heart_rate",
         "average_hrv","lowest_heart_rate","restless_periods","bedtime_start","bedtime_end",
         "time_in_bed","latency","awake_time"], None, "sleep")
    RD = load_folder(str(OURA_RAW_DIR / "dailyreadiness"),
        ["story_id","day","score","temperature_deviation","temperature_trend_deviation"],
        {"score":"oura_readiness_score"}, "readiness")
    ST = load_folder(str(OURA_RAW_DIR / "dailystress"),
        ["story_id","day","recovery_high","stress_high"], None, "stress")
    SP = load_folder(str(OURA_RAW_DIR / "dailyspo2"),
        ["story_id","day","spo2_percentage","breathing_disturbance_index"], None, "spo2")

    # spo2_percentage is exported as a dict-like string; pull out the average
    if not SP.empty and "spo2_percentage" in SP.columns:
        import ast
        def _spo2_avg(v):
            if isinstance(v, (int, float, np.number)): return float(v)
            try:
                d = ast.literal_eval(str(v))
                if isinstance(d, dict):
                    if "average" in d: return float(d["average"])
                    for x in d.values():
                        if isinstance(x, dict) and "average" in x: return float(x["average"])
            except Exception:
                pass
            return np.nan
        SP["spo2_percentage"] = SP["spo2_percentage"].apply(_spo2_avg)
        _n_bdi = SP["breathing_disturbance_index"].notna().mean()
        print(f"    spo2: breathing_disturbance_index present on "
              f"{100*_n_bdi:.1f}% of loaded rows")

    # de-duplicate sleep EPISODES before summing
    print("\n  Sleep episode de-duplication")
    n0 = len(SL)
    has_bt   = SL["bedtime_start"].notna()
    SL = pd.concat([SL[has_bt].drop_duplicates(subset=["story_id","day","bedtime_start"]),
                    SL[~has_bt].drop_duplicates()], ignore_index=True)
    print(f"    episodes {n0:,} -> {len(SL):,}  ({n0-len(SL):,} duplicates removed)")

    # sleep features
    # LOCAL WALL-CLOCK TIME (keeping participant's own UTC offset)
    def _local_wall(s):
        utc  = pd.to_datetime(s, errors="coerce", utc=True)
        off  = s.astype(str).str.extract(r"([+-])(\d{2}):(\d{2})\s*$")
        sign = off[0].map({"+": 1, "-": -1}).astype(float).fillna(0)
        mins = (sign * (off[1].astype(float) * 60 + off[2].astype(float))).fillna(0)
        return utc.dt.tz_localize(None) + pd.to_timedelta(mins, unit="m")

    for c in ["bedtime_start", "bedtime_end"]:
        SL[c] = _local_wall(SL[c])

    def mins(ts):
        return ts.hour * 60 + ts.minute + ts.second / 60.0

    def daily_sleep(g):
        out = {"total_sleep_duration": g["total_sleep_duration"].sum(),
               "light_sleep_duration": g["light_sleep_duration"].sum(),
               "deep_sleep_duration":  g["deep_sleep_duration"].sum(),
               "rem_sleep_duration":   g["rem_sleep_duration"].sum(),
               "awake_time":  g["awake_time"].sum(),
               "time_in_bed": g["time_in_bed"].sum(),
               "restless_periods": g["restless_periods"].sum(),
               "average_heart_rate": g["average_heart_rate"].mean(),
               "average_hrv": g["average_hrv"].mean(),
               "average_breath": g["average_breath"].mean(),
               "efficiency": g["efficiency"].mean(),
               "latency": g["latency"].mean(),
               "lowest_heart_rate": g["lowest_heart_rate"].min()}
        gg = g.dropna(subset=["bedtime_start","bedtime_end"])
        if gg.empty:
            out.update({"main_sleep_duration": np.nan, "main_sleep_midpoint_mins": np.nan,
                        "main_bedtime_start_mins": np.nan, "nap_count": 0,
                        "nap_total_duration": 0.0}); return pd.Series(out)
        day0 = pd.Timestamp(gg["_day"].iloc[0])
        noct_start = pd.Timestamp(day0.date()).replace(hour=18) - pd.Timedelta(days=1)
        noct_end   = pd.Timestamp(day0.date()).replace(hour=12)
        mid  = gg["bedtime_start"] + (gg["bedtime_end"] - gg["bedtime_start"]) / 2
        gg   = gg.assign(midpoint=mid)
        noct = gg[(gg.midpoint >= noct_start) & (gg.midpoint <= noct_end)]
        longs = gg[gg["type"] == "long_sleep"]
        idx = (longs["total_sleep_duration"].idxmax() if not longs.empty
               else (noct if not noct.empty else gg)["total_sleep_duration"].idxmax())
        main = gg.loc[idx]
        naps = gg[(gg.index != idx) &
                  ((gg.midpoint < noct_start) | (gg.midpoint > noct_end)) &
                  (gg["total_sleep_duration"] >= 20 * 60)]
        out.update({"main_sleep_duration": main["total_sleep_duration"],
                    "main_sleep_midpoint_mins": mins(main["midpoint"]),
                    "main_bedtime_start_mins":  mins(main["bedtime_start"]),
                    "nap_count": int(len(naps)),
                    "nap_total_duration": float(naps["total_sleep_duration"].sum())})
        return pd.Series(out)

    SL["day"] = pd.to_datetime(SL["day"], errors="coerce")
    SL["_day"] = SL["day"]
    sleep_daily = (SL.groupby(["story_id","day"]).apply(daily_sleep, include_groups=False)
                     .reset_index())
    sleep_daily = sleep_daily.drop(columns=[c for c in ["_day"] if c in sleep_daily.columns])
    print(f"    daily sleep rows: {len(sleep_daily):,}")

    # collapse each table, then merge with fan-out guards
    def one_per_day(d, label, prefer=None):
        if d.empty: return d
        d = d.copy()
        d["day"] = pd.to_datetime(d["day"], errors="coerce")
        if prefer and prefer in d.columns:
            d = d.sort_values(prefer)
        else:
            d = d.sort_values(["story_id","day"])
        n0 = len(d)
        d = d.drop_duplicates(subset=["story_id","day"], keep="first")
        print(f"    {label:<12} {n0:>7,} -> {len(d):>7,}")
        return d

    print("\n  Collapsing to one row per participant-day")
    A  = one_per_day(A,  "activity", "non_wear_time")
    DS = one_per_day(DS, "dailysleep")
    RD = one_per_day(RD, "readiness")
    ST = one_per_day(ST, "stress")
    SP = one_per_day(SP, "spo2")
    sleep_daily = one_per_day(sleep_daily, "sleep")

    print("\n  Merging (row count must not change)")
    out = A.drop(columns=[c for c in ["name"] if c in A.columns]).copy()
    print(f"    start (activity)   {len(out):>7,}")
    for nm, fr in [("sleep", sleep_daily), ("dailysleep", DS), ("readiness", RD),
                   ("stress", ST), ("spo2", SP)]:
        if fr.empty: continue
        before = len(out)
        out = out.merge(fr, on=["story_id","day"], how="left")
        assert len(out) == before, f"FAN-OUT when merging {nm} — stop and investigate"
        print(f"    + {nm:<16} {len(out):>7,}")

    out["story_id"] = pd.to_numeric(out["story_id"], errors="coerce").astype("Int64")
    OURA_DAILY_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OURA_DAILY_CSV, index=False)
    print(f"\n  Saved: {OURA_DAILY_CSV}")
    print(f"  {len(out):,} participant-days, {out['story_id'].nunique()} participants")
    return out

if REBUILD_OURA_FROM_RAW:
    _ = rebuild_oura_from_raw()
    print("\nNow set REBUILD_OURA_FROM_RAW = False in the config cell and re-run from the top.")
else:
    print("Appendix A skipped (REBUILD_OURA_FROM_RAW is False).")
    print("Set REBUILD_OURA_FROM_RAW = True to regenerate the daily file.")

# Appendix B - Building `armt_scores.csv`

Run this to (re)generate the ED-15 file, set `REBUILD_ARMT_FROM_RAW = False` again.

**Folder structure on server**

```
path/to/radar_raw/
└── <radar_id>/
    ├── questionnaire_response/ED15/<YYYYMM>/<YYYYMMDD>.csv.gz
    ├── questionnaire_response/PHQ8/…      (or questionnaire_phq8/…)
    ├── questionnaire_gad7/…               (or questionnaire_response/GAD7/…)
    ├── questionnaire_response/EDMotivation/…
    └── questionnaire_response/WeightQuestionnaire/…
```

Notes:
* **ID mapping:** Merge on the cohort file (`COHORT_CSV`) → `radar_id_clean`, plus report on how many RADAR folders failed to match.
* Practice rounds flagged and excluded (ie within 7 days of onboarding).
* ED15 behaviour frequencies: `"none"`, `"n/a"`, `"no"` → **0**; free text (eg, `"daily"`) → **missing**, 0 → "behaviour absent". The count of unparseable responses is printed.

In [ ]:
import gzip

# free-text cleaning
# Answers typed by participants are kept in a private file in DATA_DIR (not in this repository).
# free_text_tokens.json keys: zero_set (answers meaning 0), number_words (word -> number),
# weight_null (weight answers meaning missing). Only needed when rebuilding from raw data.
ZERO_SET, WORDS, WEIGHT_NULL = set(), {}, set()
if REBUILD_ARMT_FROM_RAW:
    with open(DATA_DIR / "free_text_tokens.json", encoding="utf-8") as _f:
        _tok = json.load(_f)
    ZERO_SET, WORDS, WEIGHT_NULL = set(_tok["zero_set"]), _tok["number_words"], set(_tok["weight_null"])
STONE_KG, LB_KG = 6.35029, 0.453592

# ED-15 behavioural items. Items asked as DAYS in the past week are capped at 7
# and flagged; items asked as a COUNT of episodes are left uncapped.
BEHAVIOURAL = {10: ("ed15_binge",     "times"),
               11: ("ed15_vomit",     "times"),
               12: ("ed15_laxatives", "days"),
               13: ("ed15_restrict",  "days"),
               14: ("ed15_exercise",  "days")}
EATING_IDX = [0, 2, 6, 7]          # items 1, 3, 7, 8
SHAPE_IDX  = [1, 3, 4, 5, 8, 9]    # items 2, 4, 5, 6, 9, 10

def clean_behavioural(value, unit):
    """Returns (value, capped_flag, note)."""
    s = str(value).strip().lower()
    if re.fullmatch(r"n[\s/\-]{0,3}[aq]?", s):     return 0.0, False, "typo_na->0"
    if s in ZERO_SET:                              return 0.0, False, "ok"
    if ("every" in s and "day" in s) or "daily" in s or "everyday" in s:
        return 7.0, False, "everyday->7"
    if s in WORDS:
        num, note = float(WORDS[s]), "word->number"
    else:
        found = re.findall(r"\d+(?:\.\d+)?", s)
        if not found:                              return np.nan, False, f"unreadable:{value!r}"
        num, note = float(found[0]), "ok"
    if unit == "days" and num > 7:                 return 7.0, True, f"impl_days:{num}"
    return num, False, note

def clean_weight(value):
    """Returns (kg, note). Handles stone, stone+pounds, pounds, ranges, bounds."""
    if pd.isna(value): return np.nan, "missing"
    t = str(value).strip().lower()
    if t in WEIGHT_NULL: return np.nan, "null_like"
    nums = [float(x) for x in re.findall(r"\d+(?:\.\d+)?", t)]
    if not nums: return np.nan, "unparsed"
    is_range = bool(re.search(r"\d+\s*-\s*\d+", t)) and len(nums) >= 2
    is_stone = bool(re.search(r"stone|\d\s*st\b|\bst\b|\bst\d", t))
    is_lb    = bool(re.search(r"lbs?\b|pound|\d\s*lb", t))
    if is_range:
        base = (nums[0] + nums[1]) / 2
        kg = base*STONE_KG if is_stone else base*LB_KG if is_lb else base
    elif is_stone: kg = nums[0]*STONE_KG + (nums[1]*LB_KG if len(nums) > 1 else 0.0)
    elif is_lb:    kg = nums[0]*LB_KG
    else:          kg = nums[0]
    return round(kg, 1), "ok"

def rebuild_armt_from_raw():
    section("Appendix B - rebuilding armt_scores.csv from RADAR")

    def _read_gz(p):
        try:
            with gzip.open(p, "rt", encoding="utf-8") as f:
                return pd.read_csv(f)
        except Exception:
            return None

    def _sum_score(p, n_items):
        """Sum only if ALL items were answered; otherwise missing."""
        d = _read_gz(p)
        if d is None or d.empty: return np.nan, 0
        cols = [f"value.answers.{i}.value" for i in range(n_items)]
        v = pd.to_numeric(d.loc[d.index[0]].reindex(cols), errors="coerce")
        n_ans = int(v.notna().sum())
        if REQUIRE_COMPLETE_GAD_PHQ and n_ans < n_items: return np.nan, n_ans
        return float(v.sum(skipna=True)), n_ans

    def _find(rid, subpaths, date, tol=None):
        """Companion questionnaires are three days either side."""
        tol = COMPANION_FILE_TOL_DAYS if tol is None else tol
        for delta in sorted(range(-tol, tol + 1), key=abs):
            d = date + pd.Timedelta(days=int(delta))
            for sp in subpaths:
                p = RADAR_RAW_DIR / rid / Path(*sp) / d.strftime("%Y%m") / \
                    (d.strftime("%Y%m%d") + ".csv.gz")
                if p.exists(): return p
        return None

    ATT = [f"value.answers.{i}.value" for i in range(10)]

    from collections import Counter
    records, n_files, n_bad, n_multirow, notes = [], 0, 0, 0, Counter()

    # resume from a checkpoint if a previous run was interrupted
    _bk = ED15_CSV.with_name(ED15_CSV.stem + ".backup.csv")
    done = set()
    if _bk.exists():
        _prev = pd.read_csv(_bk)
        records = _prev.to_dict("records")
        done = set(_prev["radar_id"].astype(str))
        print(f"  resuming from checkpoint: {len(records):,} records, "
              f"{len(done)} participants already done")
    folders = sorted(d.name for d in RADAR_RAW_DIR.iterdir()
                     if d.is_dir() and not d.name.startswith("."))
    print(f"  RADAR participant folders: {len(folders)}")

    for k, rid in enumerate(folders):
        if rid in done: continue
        ed15_dir = RADAR_RAW_DIR / rid / "questionnaire_response" / "ED15"
        if not ed15_dir.is_dir(): continue
        for month in sorted(p for p in ed15_dir.iterdir() if p.is_dir()
                            and not p.name.startswith(".")):
            for fp in sorted(month.glob("*.csv.gz")):
                d = _read_gz(fp)
                if d is None or d.empty: n_bad += 1; continue
                if len(d) > 1: n_multirow += 1
                try:
                    date = pd.to_datetime(fp.name.replace(".csv.gz",""), format="%Y%m%d")
                except Exception:
                    n_bad += 1; continue
                n_files += 1
                r0  = d.index[0]
                att = pd.to_numeric(d.loc[r0].reindex(ATT), errors="coerce")
                n_att = int(att.notna().sum())

                # attitudinal items must be 0-6; flag out-of-range, do not alter
                bad = att[att.notna() & ((att < 0) | (att > 6))]
                if len(bad):
                    notes["impl_attitudinal"] += len(bad)

                beh, beh_flags = {}, {}
                for idx, (nm, unit) in BEHAVIOURAL.items():
                    col = f"value.answers.{idx}.value"
                    val, capped, note = (clean_behavioural(d.loc[r0, col], unit)
                                         if col in d.columns else (np.nan, False, "absent"))
                    beh[nm] = val
                    if unit == "days": beh_flags[nm + "_capped"] = capped
                    notes[note.split(":")[0]] += 1
                gp = _find(rid, [("questionnaire_gad7",), ("questionnaire_response","GAD7")], date)
                pp = _find(rid, [("questionnaire_phq8",), ("questionnaire_response","PHQ8")], date)
                gad, gad_m = _sum_score(gp, 7) if gp else (np.nan, np.nan)
                phq, phq_m = _sum_score(pp, 8) if pp else (np.nan, np.nan)

                mot = rdy = wt = np.nan
                mp = _find(rid, [("questionnaire_response","EDMotivation")], date)
                if mp:
                    dm = _read_gz(mp)
                    if dm is not None and not dm.empty:
                        mot = pd.to_numeric(dm.loc[dm.index[0]].get("value.answers.0.value"),
                                            errors="coerce")
                        rdy = pd.to_numeric(dm.loc[dm.index[0]].get("value.answers.1.value"),
                                            errors="coerce")
                wp = _find(rid, [("questionnaire_response","WeightQuestionnaire")], date)
                if wp:
                    dw = _read_gz(wp)
                    if dw is not None and not dw.empty:
                        wt, wnote = clean_weight(dw.loc[dw.index[0]].get("value.answers.0.value"))
                        if wnote not in ("ok", "missing", "null_like"):
                            notes[f"weight_{wnote.split(':')[0]}"] += 1

                rec = {"radar_id": rid, "date": date,
                    "ed15_total_score": (round(att.mean(skipna=True), 2)
                                         if n_att >= ED15_MIN_ATTITUDINAL else np.nan),
                    "ed15_n_attitudinal": n_att,
                    "ed15_att_range_flag": bool(len(bad)),
                    "ed15_eating_concerns": (round(att.iloc[EATING_IDX].mean(), 2)
                                             if att.iloc[EATING_IDX].notna().all() else np.nan),
                    "ed15_weight_shape_concerns": (round(att.iloc[SHAPE_IDX].mean(), 2)
                                             if att.iloc[SHAPE_IDX].notna().all() else np.nan),
                    "gad7_score": gad, "gad7_n": gad_m,
                    "phq8_score": phq, "phq8_n": phq_m,
                    "motivation_score": mot, "readiness_score": rdy,
                    "weight_kg": wt, **beh, **beh_flags}
                for i in range(10):
                    rec[f"ed15_item{i+1}"] = att.iloc[i]
                records.append(rec)

        if k and k % 25 == 0:
            ED15_CSV.parent.mkdir(parents=True, exist_ok=True)
            pd.DataFrame(records).to_csv(_bk, index=False)
            print(f"    checkpoint {k}/{len(folders)}  ({len(records):,} records)", flush=True)

    raw = pd.DataFrame(records)
    if raw.empty:
        print("  No ED-15 files found - check RADAR_RAW_DIR and the VPN."); return None

    print(f"\n  ED-15 files parsed : {n_files:,}   unreadable: {n_bad}   multi-row: {n_multirow}")
    print(f"  RADAR ids with data: {raw['radar_id'].nunique()}")
    print("\n  free-text cleaning:")
    for key, cnt in notes.most_common():
        print(f"    {key:<24} {cnt:>7,}")
    for _idx, (nm, unit) in BEHAVIOURAL.items():
        if nm in raw.columns:
            n_na = int(raw[nm].isna().sum())
            extra = ""
            if unit == "days" and nm + "_capped" in raw.columns:
                extra = f" | {int(raw[nm+'_capped'].fillna(False).sum()):,} capped at 7 days"
            if n_na or extra:
                print(f"    {nm:<16} {n_na:>5} unreadable -> missing{extra}")

    # map radar_id -> story_id via the cohort file
    cohort = pd.read_csv(COHORT_CSV, low_memory=False)
    if "radar_id_clean" not in cohort.columns:
        raise KeyError("'radar_id_clean' not found in the cohort file. Columns containing "
                       f"'radar': {[c for c in cohort.columns if 'radar' in c.lower()]}")
    _norm = lambda s: s.astype(str).str.strip().str.lower()
    cmap = cohort.loc[cohort["radar_id_clean"].notna(), ["story_id", "radar_id_clean"]].copy()
    cmap["_k"] = _norm(cmap["radar_id_clean"])
    n_dup = int(cmap["_k"].duplicated().sum())
    if n_dup:
        print(f"  !! {n_dup} duplicate radar_id_clean values in the cohort file - first kept")
    cmap = cmap.drop_duplicates("_k")

    raw["_k"] = _norm(raw["radar_id"])
    out = raw.merge(cmap[["_k", "story_id"]], on="_k", how="left")
    unmatched = out[out["story_id"].isna()]
    print(f"\n  RADAR ids not in the cohort file: {unmatched['radar_id'].nunique()} "
          f"({len(unmatched):,} rows) -> dropped")
    out = out[out["story_id"].notna()].copy()
    out["story_id"] = out["story_id"].astype(int)

    # onboarding date and practice rounds
    if "onboarding_date" in cohort.columns:
        ob = cohort[["story_id", "onboarding_date"]].drop_duplicates("story_id").copy()
        ob["onboarding_date"] = pd.to_datetime(ob["onboarding_date"], format="mixed",
                                               dayfirst=True, errors="coerce")
        out = out.merge(ob, on="story_id", how="left")
        out["days_since_onboarding"] = (out["date"] - out["onboarding_date"]).dt.days
        out["practice_round"] = out["days_since_onboarding"] <= PRACTICE_ROUND_DAYS
        n_pr = int(out["practice_round"].fillna(False).sum())
        print(f"  Practice-round completions (<= {PRACTICE_ROUND_DAYS} days post-onboarding): {n_pr:,}")
        if EXCLUDE_PRACTICE_ROUNDS:
            out = out[~out["practice_round"].fillna(False)].copy()
            print("     -> EXCLUDED. Set EXCLUDE_PRACTICE_ROUNDS = False to keep them.")
        else:
            print("     -> KEPT (flagged in the `practice_round` column).")
    else:
        print("  !! no 'onboarding_date' in the cohort file - practice rounds not identified")

    n0 = len(out)
    out = out.drop_duplicates(subset=["story_id", "date"], keep="first")
    print(f"  Duplicate (story_id, date) rows removed: {n0 - len(out)}")

    out = out.sort_values(["story_id", "date"]).reset_index(drop=True)
    out["ed15_n_completions"] = out.groupby("story_id")["date"].transform("size")
    out["delta_ed15_total"]   = out.groupby("story_id")["ed15_total_score"].diff()
    out = out.drop(columns=["_k"])

    ED15_CSV.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(ED15_CSV, index=False)
    print(f"\n  Saved: {ED15_CSV}")
    print(f"  {len(out):,} completions, {out['story_id'].nunique()} participants")
    _per = out.groupby("story_id").size()
    print(f"  Completions per participant: median {_per.median():.0f} "
          f"(range {_per.min()}-{_per.max()})")
    print(f"  Participants with >= {MIN_N_ED15} completions: {int((_per >= MIN_N_ED15).sum())}")
    return out

if REBUILD_ARMT_FROM_RAW:
    _ = rebuild_armt_from_raw()
    print("\nNow set REBUILD_ARMT_FROM_RAW = False in the config cell and re-run from the top.")
else:
    print("Appendix B skipped (REBUILD_ARMT_FROM_RAW is False).")
    print("Set REBUILD_ARMT_FROM_RAW = True to rebuild armt_scores.csv.")

# Appendix C - Build analysis datasets

Writes the data this analysis actually used as `pilot_baseline.csv` (demographics, clinical characteristics) and `pilot_windows.csv` (window-level rows per participant), joined on `participant`.

**De-identification**: With `EXPORT_DEIDENTIFY = True` the exported files contain *no story_id and no calendar dates*, participants are relabelled `P001`, `P002`… and each completion is expressed as days since first completion. 


Raw daily Ōura records and raw ED-15 free-text responses are not exported.

In [ ]:
section("Appendix C - exporting the analysis datasets")

EXPORT_DEIDENTIFY = True        # relabel participants, drop calendar dates
EXPORT_WINDOW     = PRIMARY_WINDOW

_src = datasets[EXPORT_WINDOW].copy()
print(f"  Source: datasets['{EXPORT_WINDOW}']  "
      f"({len(_src):,} windows, {_src['story_id'].nunique()} participants)\n")

# 1. participant-level file
_id_cols  = ["story_id", "group"] + [c for c in BASE_NUM + BASE_CAT if c in _src.columns]
_baseline = _src.drop_duplicates("story_id")[_id_cols].copy()

_counts = (_src.groupby("story_id")
               .agg(n_windows=("date", "size"),
                    first_completion=("date", "min"),
                    last_completion=("date", "max"),
                    median_valid_days=("n_valid_days", "median"))
               .reset_index())
_counts["followup_weeks"] = ((_counts["last_completion"] - _counts["first_completion"])
                             .dt.days / 7).round(1)
_baseline = _baseline.merge(_counts, on="story_id", how="left")

# 2. window-level file
_keep_meta = (["story_id", "date"]
              + [c for c in OUTCOMES if c in _src.columns]
              + [c for c in COVARIATES + ["max_gap_days"] if c in _src.columns]
              + [c for c in CHANGE_COVARIATES + ["ed15_delta"] if c in _src.columns])
_feat_cols = [c for c in _src.columns
              if any(c.startswith(f + "_") for f in FEATURES) ]
_windows = _src[list(dict.fromkeys(_keep_meta + _feat_cols))].copy()

# 3. de-identify
if EXPORT_DEIDENTIFY:
    _codes = {sid: f"P{i:03d}" for i, sid in
              enumerate(sorted(_src["story_id"].dropna().unique()), start=1)}
    _first = _src.groupby("story_id")["date"].min()

    _windows["participant"]  = _windows["story_id"].map(_codes)
    _windows["day_in_study"] = (_windows["date"] - _windows["story_id"].map(_first)).dt.days
    _windows = _windows.drop(columns=["story_id", "date"])
    _windows = _windows[["participant", "day_in_study"]
                        + [c for c in _windows.columns
                           if c not in ("participant", "day_in_study")]]

    _baseline["participant"] = _baseline["story_id"].map(_codes)
    _key = _baseline[["participant", "story_id"]].copy()
    _baseline = _baseline.drop(columns=["story_id", "first_completion", "last_completion"])
    _baseline = _baseline[["participant"]
                          + [c for c in _baseline.columns if c != "participant"]]
    _join = "participant"
else:
    _key, _join = None, "story_id"

# 4. write
OUT_DIR.mkdir(parents=True, exist_ok=True)
_baseline.to_csv(OUT_DIR / "pilot_baseline.csv", index=False)
_windows.to_csv(OUT_DIR / "pilot_windows.csv",  index=False)

print(f"  pilot_baseline.csv : {len(_baseline):>5} rows x {_baseline.shape[1]:>4} columns")
print(f"  pilot_windows.csv  : {len(_windows):>5} rows x {_windows.shape[1]:>4} columns")
print(f"  joined on          : '{_join}'")
print(f"  written to         : {OUT_DIR}\n")

if EXPORT_DEIDENTIFY:
    _key.to_csv(OUT_DIR / "pilot_id_key_DO_NOT_SHARE.csv", index=False)
    note("Exports contain no story_id and no calendar dates. Completions are expressed as "
         "days since each participant's first completion.")
    print("  !! pilot_id_key_DO_NOT_SHARE.csv maps codes back to story_id. "
          "Keep it; do not share it.\n")
else:
    note("EXPORT_DEIDENTIFY is False: both files carry story_id and real dates. "
         "Treat them as identifiable participant data.")

print("  Columns in pilot_baseline.csv:")
print("   ", list(_baseline.columns))
print(f"\n  Columns in pilot_windows.csv: {len(_windows.columns)} total, first 12:")
print("   ", list(_windows.columns)[:12], "...")